In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import pickle

import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

from pandas.tseries.offsets import MonthEnd, MonthBegin
from maricovault.MaricoDB import MaricoSnowflake

from joblib import Parallel, delayed

In [2]:
def get_dbconnection(db_name): 

    KEY_VAULT_NAME = "prod-pwd"
    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection

In [3]:
dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


### Helper Functions

In [4]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

realignment_df = realignment_df[
    realignment_df['channel'].isin(['ECOM', 'ECOM B2C', 'ALL'])]


def realign_pskus(data, column):
    realignment_data = realignment_df.copy()
    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']

    
    data[column] = data[column].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data[column] == old_psku, column
        ] = new_psku

    return data

In [5]:
realignment_df

,psku realignment master,parent material_code old,asm,channel,new parent_material code,deletion indicator?,psku old,psku new
1,726067_ALL_E-Commerce,726067_SMO CLS MSL 500G MILLET MT ECOM B2C,ALL,ECOM,730975_SMO CLS MSL 550G MILLET MT ECOM,None,726067,730975
8,726065_ALL_E-Commerce,726065_SMO VEG TWST 500G MILLET MT ECOM,ALL,ECOM,731006_SMO VEG TWST 550G MILLET MT ECOM,None,726065,731006
9,726066_ALL_E-Commerce,726066_SMO PEP TOM 500G MILLET MT ECOM,ALL,ECOM,731008_SMO PEP TOM 550G MILLET MT ECOM,None,726066,731008
10,718574_ALL_E-Commerce,718574_SMO MSL&COR 500g PCH,ALL,ECOM,731007_SMO MAS COR 550G MILLET MT ECOM,None,718574,731007
135,718588_ALL_E-Commerce,718588_SETWET HAIRGEL ULTIMATE HOLD 250ml JAR,ALL,ECOM,730277_SW SPORTS EXTREM 250ML MT ECOM NF,None,718588,730277
...,...,...,...,...,...,...,...,...
427,725043_ALL_E-Commerce,725043_NHR NSSB 175ML BOT PARENT,ALL,ECOM,725000_NHR NSSA 175ML BOT PARENT,None,725043,725000
428,721061_ALL_E-Commerce,721061_PA ALOE LIGHT 150ML,ALL,ECOM,718617_PAR ADV ALOEVERA ENRICHED CN 150ml BTL,None,721061,718617
430,733313_ALL_E-Commerce,733313_PA EXT GOLD VIT E FT MT 475ML,ALL,ECOM,735652_PA EXT GOLD VIT E OIL 475ML,None,733313,735652
440,718310_ALL_E-Commerce,718310_PCNO 500ml JAR,ALL,ECOM,718826_PCNO 600ml JAR,None,718310,718826


In [6]:
from tqdm import tqdm

def impute_missing_dates(
        df,
        freq='M',
        key=['CHAIN', 'PARENT_MATERIAL_CODE'], 
        date_col='MONTH_DATE',
        max_date='2026-12-31'
    ):

    impute_df = df.copy()
    impute_df[date_col] = pd.to_datetime(impute_df[date_col])
    impute_df['key'] = impute_df[key].astype(str).agg('_'.join, axis=1)
    min_dates_df = impute_df.groupby(
        'key', as_index=False
    )[date_col].min()

    def impute_missing_dates_key(key, min_date, max_date):
        df_imputed = pd.DataFrame(
            pd.date_range(min_date, max_date, freq=freq),
            columns=[date_col]
        )
        df_imputed['key'] = key
        return df_imputed
    
    outputs = Parallel(n_jobs=-1)(
        delayed(impute_missing_dates_key)(row['key'], row[date_col], max_date)
        for idx, row in tqdm(min_dates_df.iterrows())
    )
    df_full = pd.concat(outputs)
    df_full.reset_index(drop=True, inplace=True)

    df_full = df_full.merge(
        impute_df, on=['key', date_col], how='left',
    )

    return df_full

### Primary Actuals + Sec Plan Data

In [7]:
plan_actuals_query = """
SELECT
    CASE 
        WHEN MCM.chain = 'Big basket B2C' THEN 'Big Basket'
        WHEN MCM.chain = 'Flipkart-National' THEN 'Flipkart National'
        WHEN MCM.chain = 'Flipkart-Minutes' THEN 'Flipkart National'
        WHEN MCM.chain = 'Nykaa' THEN 'Nykaa'
        WHEN MCM.chain = 'Myntra' THEN 'Myntra'
        WHEN MCM.chain = 'MYNTRA' THEN 'Myntra'
        WHEN MCM.chain = 'Amazon B2C' THEN 'Amazon ARIPL'
        WHEN MCM.chain = 'ARIPL' THEN 'Amazon ARIPL'
        WHEN MCM.chain = 'RK WORLDINFOCOM' THEN 'Amazon RK'
        WHEN MCM.chain = 'RKWorld' THEN 'Amazon RK'
        WHEN MCM.chain = 'Flipkart-Grocery' THEN 'Flipkart Grocery'
        WHEN MCM.chain = 'FlipkartGrocery' THEN 'Flipkart Grocery'
        WHEN MCM.chain = 'Firstcry' THEN 'First Cry'
        WHEN MCM.chain = 'CITIMALL' THEN 'City Mall'
        WHEN MCM.chain = 'Meesho' THEN 'Meesho'
        ELSE MCM.chain
    END AS chain,
    MM.parent_material_code,
    MM.material_group_code,
    MESR.month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain_type = 'E Com B2C' AND 
        chain IN ('Flipkart-National', 'Flipkart-Grocery', 'Big basket B2C', 'RK WORLDINFOCOM', 'Amazon B2C', 
        'FATEHPURIA HYGIENE', 'Nykaa', 'Flipkart-Minutes', 'Purplle', 'Myntra', 'Dealshare', 'FlipkartGrocery', 'City Mall', '1MG', 
        'ARIPL', 'First Cry', 'Meesho', 'RKWorld', 'CITIMALL', 'Firstcry', 'EMAZING DEALS', 'MYNTRA')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
WHERE
    MESR.month_date > '2022-12-31'
GROUP BY 1, 2, 3, 4
ORDER BY 1, 3, 2, 4
"""

plan_actuals_df = pd.read_sql(
    plan_actuals_query,
    prod_conn
)

In [8]:
top_chains = ['Amazon ARIPL', 'Amazon RK', 'Big Basket', 'Flipkart Grocery',
       'Flipkart National', 'Myntra', 'Nykaa','Meesho']

In [9]:
plan_actuals_df['CHAIN'].unique()

array(['1MG', 'Amazon ARIPL', 'Amazon RK', 'Big Basket', 'City Mall',
       'Dealshare', 'EMAZING DEALS', 'FATEHPURIA HYGIENE', 'First Cry',
       'Flipkart Grocery', 'Flipkart National', 'Meesho', 'Myntra',
       'Nykaa', 'Purplle'], dtype=object)

In [10]:
plan_actuals_df.columns = plan_actuals_df.columns.str.lower()
plan_actuals_df['month_date'] = pd.to_datetime(plan_actuals_df['month_date'])

In [11]:
plan_actuals_df.duplicated(subset=['chain', 'parent_material_code', 'material_group_code', 'month_date']).sum()

0

In [12]:
plan_actuals_df = realign_pskus(plan_actuals_df.copy(), column='parent_material_code')

In [13]:
plan_actuals_df.duplicated(subset=['chain', 'parent_material_code', 'material_group_code', 'month_date']).sum()

6660

In [14]:
plan_actuals_df.columns

Index(['chain', 'parent_material_code', 'material_group_code', 'month_date',
       'pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum',
       'sec_actuals_vol_rum'],
      dtype='object')

In [15]:
material_master_df = pd.read_sql(
    """select * from mst_material 
    where latest_record_ind=1 and company_code='MIL'""",
    prod_conn
)
material_master_df.columns = material_master_df.columns.str.lower()
assert material_master_df.duplicated(
    subset=['company_code', 'material_code']).sum() == 0

material_master_df['material_code'] = material_master_df['material_code'].astype(np.int64)
material_master_df.duplicated(subset=['material_code', 'parent_material_code', 'material_group_code']).sum()

0

In [16]:
plan_actuals_df = plan_actuals_df.groupby(
    ['chain', 'parent_material_code', 'month_date'],
    as_index=False
)[['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum',
       'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']].sum()

# material_master_df[['parent_material_code', 'brand_code']].dtypes
material_master_df['parent_material_code'] = material_master_df['parent_material_code'].astype(int)
material_master_df.loc[material_master_df['parent_material_code'].isin([725930,731857]), 'material_group_code'] = 'H&C_ALMND'

# offtake_df.drop(columns = ['brand_code'],inplace = True)
len_before_merge = len(plan_actuals_df)
plan_actuals_df = plan_actuals_df.merge(
    material_master_df[['parent_material_code', 'material_group_code']].drop_duplicates(),
    left_on=['parent_material_code'],right_on = ['parent_material_code'],
    how='left'
)
assert len_before_merge == len(plan_actuals_df)

In [17]:
plan_actuals_df['month_date'] = plan_actuals_df['month_date'] + MonthEnd(0)

In [18]:
plan_actuals_df = plan_actuals_df[
    (plan_actuals_df['parent_material_code'] != 715096)
]

In [19]:
715096 in plan_actuals_df['parent_material_code'].values

False

In [20]:
plan_actuals_df = plan_actuals_df.groupby(
    ['chain', 'parent_material_code', 'material_group_code', 'month_date'], as_index=False, dropna=False
).sum()

In [21]:
plan_actuals_df.shape

(148789, 8)

In [22]:
plan_actuals_df = impute_missing_dates(
    plan_actuals_df.copy(),
    key=['chain', 'parent_material_code'],
    date_col='month_date'
)

10593it [00:02, 3697.64it/s]


In [23]:
cols = ['chain', 'parent_material_code', 'material_group_code']

plan_actuals_df[cols] = plan_actuals_df.groupby('key')[cols].transform(lambda x: x.ffill().bfill())

In [24]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    plan_actuals_df[col] = plan_actuals_df[col].fillna(0)

In [25]:
plan_actuals_df['parent_material_code'] = plan_actuals_df['parent_material_code'].astype(int)

In [26]:
plan_actuals_df.duplicated(subset=['key', 'month_date']).sum()

0

In [27]:
plan_actuals_df.shape

(365486, 9)

In [28]:
# # For duplicates, keep only the row with PABABY_ML material_group_code
# duplicates = plan_actuals_df[plan_actuals_df.duplicated(subset=['key', 'month_date'], keep=False)]

# # Get indices of duplicates that are NOT PABABY_ML
# indices_to_drop = duplicates[duplicates['material_group_code'] != 'PABABY_ML'].index

# # Remove those rows
# plan_actuals_df = plan_actuals_df.drop(indices_to_drop)

# # Verify no duplicates remain
# print(plan_actuals_df.duplicated(subset=['key', 'month_date']).sum())

In [29]:
plan_actuals_df.shape

(365486, 9)

In [30]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if plan_actuals_df[col].min() < 0:
        print(col)

pri_actuals_vol_rum
sec_actuals_vol_rum


In [31]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    plan_actuals_df[col] = plan_actuals_df[col].clip(lower=0)

In [32]:
plan_actuals_df.duplicated(subset=['key', 'month_date']).sum()

0

In [33]:
plan_actuals_df.sort_values(['key', 'month_date'], inplace=True)

In [34]:
plan_actuals_df['Primary P3M'] = plan_actuals_df.groupby(
    ['key'], 
    as_index = False, group_keys = False
)['pri_actuals_vol_rum'].shift(1).rolling(window=3, min_periods=1).mean()

In [ ]:
others_df = plan_actuals_df[
    ~plan_actuals_df['chain'].isin(top_chains)
]
plan_actuals_df = plan_actuals_df[
    plan_actuals_df['chain'].isin(top_chains)
]

In [1178]:
print(others_df['chain'].unique())
print(plan_actuals_df['chain'].unique())

['1MG' 'City Mall' 'Dealshare' 'EMAZING DEALS' 'FATEHPURIA HYGIENE'
 'First Cry' 'Purplle']
['1MG' 'Amazon ARIPL' 'Amazon RK' 'Big Basket' 'City Mall' 'Dealshare'
 'EMAZING DEALS' 'FATEHPURIA HYGIENE' 'First Cry' 'Flipkart Grocery'
 'Flipkart National' 'Meesho' 'Myntra' 'Nykaa' 'Purplle']


In [38]:
plan_actuals_df['chain'].unique()

array(['1MG', 'Amazon ARIPL', 'Amazon RK', 'Big Basket', 'City Mall',
       'Dealshare', 'EMAZING DEALS', 'FATEHPURIA HYGIENE', 'First Cry',
       'Flipkart Grocery', 'Flipkart National', 'Meesho', 'Myntra',
       'Nykaa', 'Purplle'], dtype=object)

In [ ]:
x = plan_actuals_df.copy()
x.rename(columns = {'material_group_code':'Brand'},inplace = True)
len_before_merge = len(x)
x = x.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(columns={
        'brand_code': 'Brand',
        'qtr_ind_rate': 'Index Rate'
    }),
    on=['Brand'],
    how='left'
)
assert len_before_merge == len(x)
del len_before_merge
x.dtypes
x['plan_actuals_val'] = x['sec_actuals_vol_rum']*x['Index Rate']/10**7
x.groupby(['month_date'])['plan_actuals_val'].sum().reset_index()#[20:]

,month_date,plan_actuals_val
0,2023-05-31,0.017587
1,2023-06-30,0.012917
2,2023-07-31,0.074485
3,2023-08-31,0.142501
4,2023-09-30,0.140806
5,2023-10-31,0.296080
6,2023-11-30,0.149573
7,2023-12-31,0.251634
8,2024-01-31,0.096470
9,2024-02-29,0.093771


In [1139]:
# qtr_ind_rate_df

In [1140]:
# x['sec_actuals_vol_rum'].sum()

In [1142]:
plan_actuals_df.to_csv('plan_actuals_ok.csv')

In [1141]:
plan_actuals_df['sec_actuals_vol_rum'].sum()

12075756.394000001

### Offtakes

In [733]:
# offtakes_monthly_query = """
# SELECT
#     OTM.platform_name AS chain,
#     MM.parent_material_code,
#     MM.material_group_code,
#     OTM.month_date,
#     SUM(vol_in_rum) AS vol_in_roum
# FROM 
#     dwh_ecommplatform_offtake OTM 
# JOIN
# (
#     SELECT
#         material_code,
#         parent_material_code,
#         material_group_code,
#         uom_reporting,
#         vol_per_unit
#     FROM 
#         mst_material
#     WHERE
#         company_code='MIL' AND
#         latest_record_ind=1
# ) MM ON OTM.material_code = MM.material_code
# WHERE
#     OTM.platform_name NOT IN ('Blinkit', 'Swiggy', 'Zepto', 'Others', 'Amazon D2C', 'Flipkart D2C') AND
#     OTM.month_date > '2022-12-31' 
# GROUP BY 1, 2, 3, 4
# ORDER BY 1, 2, 3, 4
# """ 

# offtakes_monthly_df = pd.read_sql(
#     offtakes_monthly_query,
#     prod_conn
# )

In [734]:
# offtakes_monthly_df.columns = offtakes_monthly_df.columns.str.lower()

In [735]:
# offtakes_monthly_df['month_date'] = pd.to_datetime(offtakes_monthly_df['month_date'])
# offtakes_monthly_df['parent_material_code'] = offtakes_monthly_df['parent_material_code'].astype(int)

In [736]:
# offtakes_monthly_df.rename(columns={'vol_in_roum': 'offtake_vol_rum'}, inplace=True)

In [737]:
# offtakes_monthly_df

In [738]:
# offtakes_monthly_df['month_date'].max()

In [739]:
mmonth_df = pd.read_sql("""
    SELECT * 
    FROM 
        TRN_DF_ECOM_OFFTAKE_CHAIN_PSKU_NEW
    WHERE
        
        run_month='2026-08-31'
""", 
    dev_conn
)

In [740]:
mmonth_df.columns = mmonth_df.columns.str.lower()

In [741]:
mmonth_df['month_date'] = pd.to_datetime(mmonth_df['month_date'])
mmonth_df['parent_material_code'] = mmonth_df['parent_material_code'].astype(int)

In [742]:
mmonth_df.head()

,month_date,platform_name,parent_material_code,vol_in_rum,indexbpm,brand_code,imputed,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,run_month
0,2026-07-31,Amazon ARIPL,718288,33.954,47.150311,SAFF GOLD,0,0,0,0,0,1,0,0,0,0,1,NaN,3,2026-08-31
1,2026-08-31,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,0,0,0,1,0,0,0,0,1,0,NaN,3,2026-08-31
2,2026-09-30,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,1,0,0,0,0,1,0,0,0,0,0.0,3,2026-08-31
3,2026-10-31,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,0,1,0,0,0,0,1,0,0,0,0.0,4,2026-08-31
4,2026-11-30,Amazon ARIPL,718288,0.000,0.000000,SAFF GOLD,1,0,0,1,0,0,0,0,1,0,0,0.0,4,2026-08-31


In [743]:
mmonth_df = mmonth_df.groupby(
    ['platform_name', 'parent_material_code', 'brand_code', 'month_date'],
    as_index=False
)['vol_in_rum'].sum()

In [744]:
mmonth_df.rename(
    columns = {'platform_name': 'chain', 'brand_code': 'material_group_code', 'vol_in_rum': 'offtake_vol_rum'},
    inplace=True
)

In [745]:
# offtakes_monthly_df = pd.concat([
#     offtakes_monthly_df,
#     mmonth_df
# ])
offtakes_monthly_df = mmonth_df.copy()

In [746]:
offtakes_monthly_df[offtakes_monthly_df['month_date'] == '2026-07-31']['offtake_vol_rum'].sum()

385513.2369194125

In [747]:
offtakes_monthly_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()

0

In [748]:
portfolio_query = """SELECT DISTINCT
    MATERIAL_GROUP_CODE AS BRAND_CODE,
    SAP_PORTFOLIO_NAME AS PORTFOLIO
FROM MST_MATERIAL
WHERE COMPANY_CODE = 'MIL'
  AND LATEST_RECORD_IND = '1'
  AND MATERIAL_GROUP_CODE IS NOT NULL
  AND SAP_PORTFOLIO_NAME IS NOT NULL
ORDER BY SAP_PORTFOLIO_NAME, MATERIAL_GROUP_CODE;"""
brand_md_df = pd.read_sql(portfolio_query, prod_conn)
brand_md_df.columns = brand_md_df.columns.str.lower()
brand_md_df.head()

,brand_code,portfolio
0,BD_BDOL_M,BEARDO
1,BD_HRWX_M,BEARDO
2,BRD_BDCOL,BEARDO
3,BRD_BDOIL,BEARDO
4,BRD_BDSPR,BEARDO


In [749]:
len_before_merge = len(offtakes_monthly_df)
offtakes_monthly_df = offtakes_monthly_df.merge(
    brand_md_df.rename(columns={'brand_code': 'material_group_code'}),
    on=['material_group_code'],
    how='left'
)
assert len_before_merge == len(offtakes_monthly_df)

In [750]:
offtakes_monthly_df[offtakes_monthly_df['portfolio'].isna()]

,chain,parent_material_code,material_group_code,month_date,offtake_vol_rum,portfolio


In [751]:
offtakes_monthly_df['chain'] = np.where(
    (offtakes_monthly_df['chain'] == 'Amazon'),
    np.where(
        offtakes_monthly_df['portfolio'].isin(['Saffola Oils', 'Foods']),
        'Amazon ARIPL',
        'Amazon RK'
    ),
    offtakes_monthly_df['chain']
)

In [752]:
offtakes_monthly_df[offtakes_monthly_df['chain'].str.startswith('Amazon')][['chain', 'portfolio']].drop_duplicates()

,chain,portfolio
0,Amazon ARIPL,SAFFOLA OILS
88,Amazon ARIPL,FOODS
935,Amazon RK,CNO
968,Amazon RK,SAFFOLA OILS
1160,Amazon RK,HAIR OILS
1372,Amazon RK,FOODS
1425,Amazon RK,OTHERS
1559,Amazon RK,MALE GROOMING
1718,Amazon RK,PREM. HAIR NOUR.
2870,Amazon RK,SKIN CARE


In [753]:
offtakes_monthly_df['chain'].unique()

array(['Amazon ARIPL', 'Amazon RK', 'Big Basket', 'Flipkart Grocery',
       'Flipkart National', 'Meesho', 'Myntra', 'Nykaa', 'Purplle'],
      dtype=object)

In [754]:
offtakes_monthly_df['parent_material_code'] = offtakes_monthly_df['parent_material_code'].astype(int)

In [755]:
offtakes_monthly_df['key'] = offtakes_monthly_df[['chain', 'parent_material_code']].astype(str).agg('_'.join, axis=1)

In [756]:
offtakes_monthly_df

,chain,parent_material_code,material_group_code,month_date,offtake_vol_rum,portfolio,key
0,Amazon ARIPL,718288,SAFF GOLD,2026-07-31,33.954,SAFFOLA OILS,Amazon ARIPL_718288
1,Amazon ARIPL,718288,SAFF GOLD,2026-08-31,0.000,SAFFOLA OILS,Amazon ARIPL_718288
2,Amazon ARIPL,718288,SAFF GOLD,2026-09-30,0.000,SAFFOLA OILS,Amazon ARIPL_718288
3,Amazon ARIPL,718288,SAFF GOLD,2026-10-31,0.000,SAFFOLA OILS,Amazon ARIPL_718288
4,Amazon ARIPL,718288,SAFF GOLD,2026-11-30,0.000,SAFFOLA OILS,Amazon ARIPL_718288
...,...,...,...,...,...,...,...
138925,Purplle,810674,PA_ESS_HO,2027-01-31,0.000,HAIR OILS,Purplle_810674
138926,Purplle,810674,PA_ESS_HO,2027-02-28,0.000,HAIR OILS,Purplle_810674
138927,Purplle,810674,PA_ESS_HO,2027-03-31,0.000,HAIR OILS,Purplle_810674
138928,Purplle,810674,PA_ESS_HO,2027-04-30,0.000,HAIR OILS,Purplle_810674


In [757]:
offtakes_monthly_df.duplicated(subset=['key', 'month_date']).sum()

0

In [758]:
715096 in offtakes_monthly_df['parent_material_code'].values

True

In [759]:
offtakes_monthly_df = offtakes_monthly_df[
    offtakes_monthly_df['parent_material_code'] != 715096
]

In [760]:
offtakes_monthly_df['offtake_vol_rum'].min()

0.0

In [36]:
def read_qtr_ind_rate_table():
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=prod_conn, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    
    return qtr_ind_rate
qtr_ind_rate_df = read_qtr_ind_rate_table()
qtr_ind_rate_df.head()

,month_date,brand_code,qtr_ind_rate
0,2027-03-31,CMX_WELPD,3014.000
1,2027-03-31,4700_BCPC,222.000
2,2027-03-31,CMX_PRTPD,3014.000
3,2027-03-31,PA_CN_HGO,488.152
4,2027-03-31,TRU_RAWDF,800.000


In [762]:
qtr_ind_rate_df

,month_date,brand_code,qtr_ind_rate
0,2027-03-31,CMX_WELPD,3014.00000
1,2027-03-31,4700_BCPC,222.00000
2,2027-03-31,CMX_PRTPD,3014.00000
3,2027-03-31,PA_CN_HGO,488.15200
4,2027-03-31,TRU_RAWDF,800.00000
...,...,...,...
352,2018-03-31,PADV-HRAHF,0.00000
353,2018-03-31,PAR T HFS,0.00000
354,2020-03-31,LIVON HG Shampoo,0.00000
355,2020-03-31,NHO ACE,0.00000


In [763]:
# x = offtakes_monthly_df.copy()
# x.rename(columns = {'material_group_code':'Brand'},inplace = True)
# len_before_merge = len(x)
# x = x.merge(
#     qtr_ind_rate_df.drop('month_date', axis=1).rename(columns={
#         'brand_code': 'Brand',
#         'qtr_ind_rate': 'Index Rate'
#     }),
#     on=['Brand'],
#     how='left'
# )
# assert len_before_merge == len(x)
# del len_before_merge
# x['offtake_val'] = x['offtake_vol_rum']*x['Index Rate']/10**7
# x.groupby(['month_date'])['offtake_val'].sum().reset_index()[20:]

### SOH

In [764]:
soh_df = pd.read_csv('/data/aman_singh/acuuracy_check/soh_base_aug_run.csv')

In [765]:
soh_df.columns = soh_df.columns.str.lower()

In [766]:
soh_df

,chain,fsn,soh,material_code,desc,brand,uom,vol per unit,vol,fy index,...,roum divide,month,day,month.1,unnamed: 18,unnamed: 19,unnamed: 20,unnamed: 21,unnamed: 41,unnamed: 42
0,Blinkit,10000059,3131.0,707136,SAF TOTAL 5LT JAR,SAFF KO,KL,5000.0,15.655000,182263.023590,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Blinkit,10000062,500.0,722897,SMO 38G CUR&PEP HDC,SFOATS-FL,TO,38.0,0.019000,252810.051419,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Blinkit,10000063,2711.0,722898,SMO 38G MAS&COR HDC,SFOATS-FL,TO,38.0,0.103018,252810.051419,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Blinkit,10000361,1542.0,809267,SAFF SALT PLUS 1 KG PCH,SAFF SALT,TO,1000.0,1.542000,25994.993781,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Blinkit,10000362,3387.0,713333,SAF TASTY PLUS 1L PCH,SAFF KOCO,KL,1000.0,3.387000,145305.652227,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92831,Flipkart Grocery,TLCHFK7WEJ6HKNYZ,4.0,809047,PA BABY POWDER 200 GMS,PABABY_GM,KG,200.0,0.800000,366.484998,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
92832,Flipkart Minutes,TLCHFK7WEJ6HKNYZ,1062.0,809047,PA BABY POWDER 200 GMS,PABABY_GM,KG,200.0,212.400000,366.484998,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
92833,Flipkart Grocery,WIPHBWADB6FNPEJH,29.0,810688,PA BABY FACE BDY WIPE 362G,PABABY_GM,KG,362.0,10.498000,366.484998,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
92834,Flipkart Minutes,WIPHBWADB6FNPEJH,285.0,810688,PA BABY FACE BDY WIPE 362G,PABABY_GM,KG,362.0,103.170000,366.484998,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [767]:
soh_df.groupby(['chain'])['date'].max()

chain
Amazon ARIPL         2026-08-01
Amazon RK            2026-08-01
BB                   2025-03-22
Big Basket           2026-08-01
Blinkit              2026-08-01
Flipkart Grocery     2026-08-17
Flipkart Mintues     2025-10-31
Flipkart Minutes     2026-08-17
Flipkart National    2026-08-17
Flipkart-Grocery     2026-07-01
Meesho               2026-08-01
Myntra               2026-08-01
Nykaa                2026-08-01
Purplle              2025-11-17
Swiggy               2026-08-01
Zepto                2026-08-01
Name: date, dtype: object

In [768]:
import numpy as np
soh_df['date'] = np.where(
    soh_df['date'] > '2026-07-31',
    '2026-07-31',
    soh_df['date']
)

In [769]:
soh_df.groupby(['chain'])['date'].max()

chain
Amazon ARIPL         2026-07-31
Amazon RK            2026-07-31
BB                   2025-03-22
Big Basket           2026-07-31
Blinkit              2026-07-31
Flipkart Grocery     2026-07-31
Flipkart Mintues     2025-10-31
Flipkart Minutes     2026-07-31
Flipkart National    2026-07-31
Flipkart-Grocery     2026-07-01
Meesho               2026-07-31
Myntra               2026-07-31
Nykaa                2026-07-31
Purplle              2025-11-17
Swiggy               2026-07-31
Zepto                2026-07-31
Name: date, dtype: object

In [770]:
material_master = pd.read_sql("""
SELECT * 
FROM
    mst_material
WHERE
    company_code='MIL' AND
    latest_record_ind=1
""",
    prod_conn
)

In [771]:
material_master.columns = material_master.columns.str.lower()
material_master['material_code'] = material_master['material_code'].astype(int)

In [772]:
soh_df.columns

Index(['chain', 'fsn', 'soh', 'material_code', 'desc', 'brand', 'uom',
       'vol per unit', 'vol', 'fy index', 'bpm in lacs', 'category',
       'ecom brands', 'file', 'vol in kl', 'psku', 'pdes', 'date', 'asin',
       'asin.1', 'platform_name.1', 'product_title', 'ean', 'units',
       'material_group_code', 'uom_reporting', 'vol_per_unit', 'vol_in_lit',
       'vol in rum', 'indexrate', 'indexbpm', 'club sku', 'roum',
       'roum divide', 'month', 'day', 'month.1', 'unnamed: 18', 'unnamed: 19',
       'unnamed: 20', 'unnamed: 21', 'unnamed: 41', 'unnamed: 42'],
      dtype='object')

In [773]:
material_master.columns

Index(['company_code', 'material_code', 'material_desc', 'material_group_code',
       'material_group_desc', 'division_code', 'division_name', 'uom_base',
       'uom_weight', 'gross_weight', 'net_weight', 'parent_material_desc',
       'material_type', 'material_type_desc', 'uom_sales', 'uom_reporting',
       'mg1_code', 'mg1_desc', 'mg2_code', 'mg2_desc', 'mg3_code', 'mg3_desc',
       'mg4_code', 'mg4_desc', 'mg5_code', 'mg5_desc', 'profit_centre_code',
       'vol_per_unit', 'unit_per_case_nbr', 'convert_to_ton', 'convert_to_kl',
       'convert_to_l', 'convert_to_kg', 'convert_to_ml', 'convert_to_gm',
       'csd_id', 'ean_id', 'standard_cost_amt', 'source_system_id',
       'last_bi_updt_date', 'last_src_updt_date', 'material_sk',
       'effective_start_date', 'effective_end_date', 'latest_record_ind',
       'version_nbr', 'parent_material_code', 'parent_material1_code',
       'parent_material1_desc', 'shelf_life_days',
       'manual_material_group_desc', 'manual_mg1_desc',

In [774]:
for sku in soh_df['material_code'].unique():
    try:
        int(sku)
    except:
        print(sku)

Combo


In [775]:
soh_df = soh_df[soh_df['material_code'] != 'Combo']

In [776]:
soh_df['material_code'] = soh_df['material_code'].astype(int)

In [777]:
soh_df.columns

Index(['chain', 'fsn', 'soh', 'material_code', 'desc', 'brand', 'uom',
       'vol per unit', 'vol', 'fy index', 'bpm in lacs', 'category',
       'ecom brands', 'file', 'vol in kl', 'psku', 'pdes', 'date', 'asin',
       'asin.1', 'platform_name.1', 'product_title', 'ean', 'units',
       'material_group_code', 'uom_reporting', 'vol_per_unit', 'vol_in_lit',
       'vol in rum', 'indexrate', 'indexbpm', 'club sku', 'roum',
       'roum divide', 'month', 'day', 'month.1', 'unnamed: 18', 'unnamed: 19',
       'unnamed: 20', 'unnamed: 21', 'unnamed: 41', 'unnamed: 42'],
      dtype='object')

In [778]:
soh_df.drop(['material_group_code', 'vol_per_unit', 'uom_reporting'], axis=1, inplace=True)

In [779]:
len_before_merge = len(soh_df)
soh_df = soh_df.merge(
    material_master[['material_code', 'parent_material_code', 'material_group_code', 'vol_per_unit', 'uom_reporting']],
    on=['material_code'],
    how='left'
)
assert len_before_merge == len(soh_df)
del len_before_merge

In [780]:
# soh_df[soh_df['parent_material_code'].isna()]

In [781]:
soh_df = soh_df[soh_df['parent_material_code'].notna()]

In [782]:
soh_df['parent_material_code'] = soh_df['parent_material_code'].astype(int)

In [783]:
soh_df['chain'] = soh_df['chain'].replace({
    'MYNTRA': 'Myntra',
    'BB': 'Big Basket',
    'Flipkart Mintues': 'Flipkart National',
    'Flipkart Minutes': 'Flipkart National'
})

In [784]:
sorted(soh_df['chain'].unique())

['Amazon ARIPL',
 'Amazon RK',
 'Big Basket',
 'Blinkit',
 'Flipkart Grocery',
 'Flipkart National',
 'Flipkart-Grocery',
 'Meesho',
 'Myntra',
 'Nykaa',
 'Purplle',
 'Swiggy',
 'Zepto']

In [785]:
soh_df

,chain,fsn,soh,material_code,desc,brand,uom,vol per unit,vol,fy index,...,unnamed: 18,unnamed: 19,unnamed: 20,unnamed: 21,unnamed: 41,unnamed: 42,parent_material_code,material_group_code,vol_per_unit,uom_reporting
0,Blinkit,10000059,3131.0,707136,SAF TOTAL 5LT JAR,SAFF KO,KL,5000.0,15.655000,182263.023590,...,NaN,NaN,NaN,NaN,NaN,NaN,718322,SAFF KO,5000.0000,KL
1,Blinkit,10000062,500.0,722897,SMO 38G CUR&PEP HDC,SFOATS-FL,TO,38.0,0.019000,252810.051419,...,NaN,NaN,NaN,NaN,NaN,NaN,718492,SFOATS-FL,37.9997,TO
2,Blinkit,10000063,2711.0,722898,SMO 38G MAS&COR HDC,SFOATS-FL,TO,38.0,0.103018,252810.051419,...,NaN,NaN,NaN,NaN,NaN,NaN,718494,SFOATS-FL,37.9997,TO
3,Blinkit,10000361,1542.0,809267,SAFF SALT PLUS 1 KG PCH,SAFF SALT,TO,1000.0,1.542000,25994.993781,...,NaN,NaN,NaN,NaN,NaN,NaN,807029,SAFF SALT,1000.0000,TO
4,Blinkit,10000362,3387.0,713333,SAF TASTY PLUS 1L PCH,SAFF KOCO,KL,1000.0,3.387000,145305.652227,...,NaN,NaN,NaN,NaN,NaN,NaN,718328,SAFF KOCO,1000.0000,KL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92826,Flipkart Grocery,TLCHFK7WEJ6HKNYZ,4.0,809047,PA BABY POWDER 200 GMS,PABABY_GM,KG,200.0,0.800000,366.484998,...,NaN,NaN,NaN,NaN,NaN,NaN,809046,PABABY_GM,200.0000,KG
92827,Flipkart National,TLCHFK7WEJ6HKNYZ,1062.0,809047,PA BABY POWDER 200 GMS,PABABY_GM,KG,200.0,212.400000,366.484998,...,NaN,NaN,NaN,NaN,NaN,NaN,809046,PABABY_GM,200.0000,KG
92828,Flipkart Grocery,WIPHBWADB6FNPEJH,29.0,810688,PA BABY FACE BDY WIPE 362G,PABABY_GM,KG,362.0,10.498000,366.484998,...,NaN,NaN,NaN,NaN,NaN,NaN,810738,PABABY_GM,362.0000,KG
92829,Flipkart National,WIPHBWADB6FNPEJH,285.0,810688,PA BABY FACE BDY WIPE 362G,PABABY_GM,KG,362.0,103.170000,366.484998,...,NaN,NaN,NaN,NaN,NaN,NaN,810738,PABABY_GM,362.0000,KG


In [786]:
soh_df['current_soh'] = soh_df['soh'] * soh_df['vol_per_unit'] / soh_df['uom_reporting'].map(
    lambda x: (10 ** 6) if x in ('KL', 'TO') else (10 ** 3)
)

In [787]:
soh_df.dtypes

chain                    object
fsn                      object
soh                     float64
material_code             int64
desc                     object
brand                    object
uom                      object
vol per unit            float64
vol                     float64
fy index                float64
bpm in lacs             float64
category                 object
ecom brands              object
file                     object
vol in kl               float64
psku                     object
pdes                     object
date                     object
asin                     object
asin.1                   object
platform_name.1          object
product_title            object
ean                     float64
units                   float64
vol_in_lit              float64
vol in rum              float64
indexrate               float64
indexbpm                float64
club sku                 object
roum                    float64
roum divide             float64
month   

In [788]:
soh_df.rename(
    columns={'date': 'inv date'},
    inplace=True
)

In [789]:
soh_df = realign_pskus(soh_df.copy(), 'parent_material_code')

In [790]:
soh_df = soh_df.groupby(
    ['chain', 'parent_material_code', 'inv date'], as_index=False
)['current_soh'].sum().rename(columns={
    'inv date': 'as_on_date'
})

In [791]:
soh_df['key'] = soh_df[['chain', 'parent_material_code']].astype(str).agg('_'.join, axis=1)

In [792]:
soh_df['as_on_date'] = pd.to_datetime(soh_df['as_on_date'])

In [793]:
soh_df['run_month'] = soh_df['as_on_date'] + MonthEnd(0)

In [794]:
soh_df.dtypes

chain                           object
parent_material_code             int64
as_on_date              datetime64[ns]
current_soh                    float64
key                             object
run_month               datetime64[ns]
dtype: object

In [795]:
soh_df

,chain,parent_material_code,as_on_date,current_soh,key,run_month
0,Amazon ARIPL,718288,2024-12-14,16.794,Amazon ARIPL_718288,2024-12-31
1,Amazon ARIPL,718288,2025-01-25,28.038,Amazon ARIPL_718288,2025-01-31
2,Amazon ARIPL,718288,2025-02-22,23.850,Amazon ARIPL_718288,2025-02-28
3,Amazon ARIPL,718288,2025-03-29,9.204,Amazon ARIPL_718288,2025-03-31
4,Amazon ARIPL,718288,2025-04-19,7.044,Amazon ARIPL_718288,2025-04-30
...,...,...,...,...,...,...
44615,Zepto,811181,2026-07-31,0.526,Zepto_811181,2026-07-31
44616,Zepto,811287,2026-05-01,9.500,Zepto_811287,2026-05-31
44617,Zepto,811287,2026-06-01,55.600,Zepto_811287,2026-06-30
44618,Zepto,811287,2026-07-01,35.900,Zepto_811287,2026-07-31


In [796]:
date_wise_soh_vol_sum = soh_df.groupby(['as_on_date'], as_index=False)['current_soh'].sum()

In [797]:
date_wise_soh_vol_sum[date_wise_soh_vol_sum['current_soh'] == 0]

,as_on_date,current_soh


In [798]:
soh_df[
    soh_df['as_on_date'].isin(
        date_wise_soh_vol_sum[date_wise_soh_vol_sum['current_soh'] == 0]['as_on_date'].unique()
    )
]['current_soh'].sum()

0.0

In [799]:
soh_df = soh_df[
    ~soh_df['as_on_date'].isin(
        date_wise_soh_vol_sum[date_wise_soh_vol_sum['current_soh'] == 0]['as_on_date'].unique()
    )
]

In [800]:
715096 in soh_df['parent_material_code'].values

False

In [801]:
soh_df[['chain', 'as_on_date']].drop_duplicates().sort_values(by=['chain', 'as_on_date'])

,chain,as_on_date
0,Amazon ARIPL,2024-12-14
1,Amazon ARIPL,2025-01-25
2,Amazon ARIPL,2025-02-22
3,Amazon ARIPL,2025-03-29
4,Amazon ARIPL,2025-04-19
...,...,...
40472,Zepto,2026-04-01
40451,Zepto,2026-05-01
40452,Zepto,2026-06-01
40453,Zepto,2026-07-01


In [802]:
soh_df[soh_df['chain'] == 'Amazon ARIPL'].head(60)

,chain,parent_material_code,as_on_date,current_soh,key,run_month
0,Amazon ARIPL,718288,2024-12-14,16.794,Amazon ARIPL_718288,2024-12-31
1,Amazon ARIPL,718288,2025-01-25,28.038,Amazon ARIPL_718288,2025-01-31
2,Amazon ARIPL,718288,2025-02-22,23.850,Amazon ARIPL_718288,2025-02-28
3,Amazon ARIPL,718288,2025-03-29,9.204,Amazon ARIPL_718288,2025-03-31
4,Amazon ARIPL,718288,2025-04-19,7.044,Amazon ARIPL_718288,2025-04-30
5,Amazon ARIPL,718288,2025-05-31,12.048,Amazon ARIPL_718288,2025-05-31
6,Amazon ARIPL,718288,2025-06-19,9.138,Amazon ARIPL_718288,2025-06-30
7,Amazon ARIPL,718288,2025-07-31,6.840,Amazon ARIPL_718288,2025-07-31
8,Amazon ARIPL,718288,2025-08-02,17.118,Amazon ARIPL_718288,2025-08-31
9,Amazon ARIPL,718288,2025-10-05,18.864,Amazon ARIPL_718288,2025-10-31


### Forecasts

In [803]:
forecasts = pd.read_excel(
    r"/data/aman_singh/acuuracy_check/Heuristics_all_combination_ecom_aug_live.xlsx",
    sheet_name='base'
)
forecasts = forecasts[forecasts['key'].notna()]
forecasts.head()

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,pred_value_SARIMA,shrink_ratio_sarima,recency_heuristic_sarima,seasonal_heuristic_sarima,non_seasonal_heuristic_sarima,final_heuristic_sarima,final_heuristic_sarima_value,final_heuristic_sarima_value_2,ratio_last_year,quarter
0,Amazon ARIPL_718288,2026-08-31,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.000000,1.0,33.954000,33.954,33.954000,33.954,0.471503,0.471503,1.443409,3
1,Flipkart National_722208,2026-09-30,0.0,0.016667,121.598226,195.280425,0.0,0.000231,1.688577,2.711767,...,0.719459,1.0,51.809888,0.000,51.809888,0.000,0.000000,0.000000,0.000000,0
2,Amazon ARIPL_718288,2026-10-31,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.000000,1.0,33.954000,33.954,33.954000,33.954,0.471503,0.471503,1.087573,3
3,Amazon ARIPL_718288,2026-11-30,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.000000,1.0,33.954000,33.954,33.954000,33.954,0.471503,0.471503,1.266715,4
4,Amazon ARIPL_718288,2026-12-31,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.000000,1.0,33.954000,33.954,33.954000,33.954,0.471503,0.471503,1.184903,4


In [804]:
forecasts['parent_material_code'] = forecasts['parent_material_code'].astype(int)

In [805]:
forecasts.columns

Index(['key', 'month_date', 'pred_p3m', 'pred_p6m', 'pred_prophet', 'pred_rf',
       'pred_value_p3m', 'pred_value_p6m', 'pred_value_prophet',
       'pred_value_rf',
       ...
       'pred_value_SARIMA', 'shrink_ratio_sarima', 'recency_heuristic_sarima',
       'seasonal_heuristic_sarima', 'non_seasonal_heuristic_sarima',
       'final_heuristic_sarima', 'final_heuristic_sarima_value',
       'final_heuristic_sarima_value_2', 'ratio_last_year', 'quarter'],
      dtype='object', length=162)

In [806]:
forecasts['Final Heuristic Prophet Vol'] = forecasts['final Heuristic Value'] * (10 ** 7) / forecasts['qtr_ind_rate_x']
forecasts.rename(columns = {'run_month_x': 'run_month'}, inplace=True)

In [807]:
forecasts = forecasts[['key', 'month_date', 'platform_name', 'parent_material_code', 'brand_code', 
                       'run_month', 'M month', 'Final Heuristic Prophet Vol', 'skipped']]

In [808]:
forecasts['run_month'].unique()

<DatetimeArray>
['2026-08-31 00:00:00']
Length: 1, dtype: datetime64[ns]

In [809]:
forecasts.duplicated(['key', 'run_month', 'month_date']).sum()

0

In [810]:
forecasts[forecasts.duplicated(['key', 'run_month', 'month_date'])]

,key,month_date,platform_name,parent_material_code,brand_code,run_month,M month,Final Heuristic Prophet Vol,skipped


In [811]:
forecasts['platform_name'].unique()

array(['Amazon ARIPL', 'Flipkart National', 'Flipkart Grocery',
       'Big Basket', 'Amazon RK', 'Meesho', 'Myntra', 'Nykaa', 'Purplle'],
      dtype=object)

In [812]:
# forecasts.rename(columns={'Final Heuristic Prophet 2 Vol': 'Final Heuristic Prophet Vol'}, inplace=True)

In [813]:
top_chains

['Amazon ARIPL',
 'Amazon RK',
 'Big Basket',
 'Flipkart Grocery',
 'Flipkart National',
 'Myntra',
 'Nykaa',
 'Meesho']

### Collate Everything

In [814]:
print(f"SOH", soh_df['chain'].unique())
print("Offtakes:", offtakes_monthly_df['chain'].unique())
print("Plan Actuals:", plan_actuals_df['chain'].unique())

SOH ['Amazon ARIPL' 'Amazon RK' 'Big Basket' 'Blinkit' 'Flipkart Grocery'
 'Flipkart National' 'Flipkart-Grocery' 'Meesho' 'Myntra' 'Nykaa'
 'Purplle' 'Swiggy' 'Zepto']
Offtakes: ['Amazon ARIPL' 'Amazon RK' 'Big Basket' 'Flipkart Grocery'
 'Flipkart National' 'Meesho' 'Myntra' 'Nykaa' 'Purplle']
Plan Actuals: ['Amazon ARIPL' 'Amazon RK' 'Big Basket' 'Flipkart Grocery'
 'Flipkart National' 'Meesho' 'Myntra' 'Nykaa']


In [815]:
offtakes_monthly_df

,chain,parent_material_code,material_group_code,month_date,offtake_vol_rum,portfolio,key
0,Amazon ARIPL,718288,SAFF GOLD,2026-07-31,33.954,SAFFOLA OILS,Amazon ARIPL_718288
1,Amazon ARIPL,718288,SAFF GOLD,2026-08-31,0.000,SAFFOLA OILS,Amazon ARIPL_718288
2,Amazon ARIPL,718288,SAFF GOLD,2026-09-30,0.000,SAFFOLA OILS,Amazon ARIPL_718288
3,Amazon ARIPL,718288,SAFF GOLD,2026-10-31,0.000,SAFFOLA OILS,Amazon ARIPL_718288
4,Amazon ARIPL,718288,SAFF GOLD,2026-11-30,0.000,SAFFOLA OILS,Amazon ARIPL_718288
...,...,...,...,...,...,...,...
138925,Purplle,810674,PA_ESS_HO,2027-01-31,0.000,HAIR OILS,Purplle_810674
138926,Purplle,810674,PA_ESS_HO,2027-02-28,0.000,HAIR OILS,Purplle_810674
138927,Purplle,810674,PA_ESS_HO,2027-03-31,0.000,HAIR OILS,Purplle_810674
138928,Purplle,810674,PA_ESS_HO,2027-04-30,0.000,HAIR OILS,Purplle_810674


In [816]:
final_df = plan_actuals_df[
    ['key', 'chain', 'parent_material_code', 'material_group_code']
].drop_duplicates()

In [817]:
duplicates = final_df[final_df.duplicated(subset=['key'], keep=False)]
duplicates

,key,chain,parent_material_code,material_group_code


In [818]:
# For duplicates, keep only the row with PABABY_ML material_group_code
duplicates = final_df[final_df.duplicated(subset=['key'], keep=False)]

# Get indices of duplicates that are NOT PABABY_ML
indices_to_drop = duplicates[duplicates['material_group_code'] != 'PABABY_ML'].index

# Remove those rows
final_df = final_df.drop(indices_to_drop)

# Verify no duplicates remain
print(final_df.duplicated(subset=['key']).sum())

0


In [819]:
forecasts['run_month'].unique()

<DatetimeArray>
['2026-08-31 00:00:00']
Length: 1, dtype: datetime64[ns]

In [820]:
tmp_df = pd.DataFrame()

for rm in ['2026-08-31']: 
    mth_dates = [pd.to_datetime(rm) + MonthEnd(i) for i in range(-1, 9)]
    for mth_dt in mth_dates:
        tmp_df2 = final_df.copy()
        tmp_df2['run_month'] = pd.to_datetime(rm)
        tmp_df2['month_date'] = pd.to_datetime(mth_dt)

        tmp_df = pd.concat([tmp_df, tmp_df2], ignore_index=True)
        del tmp_df2

final_df = tmp_df.copy()    
del tmp_df

### Merge Plan

In [821]:
plan_actuals_df.duplicated(subset=['key', 'month_date']).sum()

0

In [822]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    plan_actuals_df[['key', 'month_date', 'pri_actuals_vol_rum', 'sec_apo_plan_vol_rum', 'Primary P3M']],
    on=['key', 'month_date'],
    how='left'
)
assert len(final_df) == len_before_merge
del len_before_merge

In [823]:
final_df.duplicated(subset=['key', 'month_date']).sum()

0

In [824]:
plan_actuals_df[plan_actuals_df['month_date']<'2026-06-30'].to_csv('/data/aman_singh/acuuracy_check/plan_actuals_df_ecom.csv', index=False)

In [825]:
final_df.sort_values(by=['run_month', 'key', 'month_date'], inplace=True)

In [826]:
for col in ['Primary P3M']:
    # if not 'LY' in col:  'LY P6M',
    final_df.loc[final_df['month_date'] > final_df['run_month'], [col]] = np.nan
    final_df[col] = final_df.groupby(['run_month', 'key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

### Merge offtakes

In [827]:
final_df['key'].nunique()

6092

In [828]:
offtakes_monthly_df['key'].nunique()

3536

In [829]:
yy = final_df.copy()

In [830]:
### Monthly Actual Offtakes
len_before_merge = len(final_df)
final_df = final_df.merge(
    offtakes_monthly_df[['month_date', 'key', 'offtake_vol_rum']].rename(columns={
        'offtake_vol_rum': 'Offtake Chain PSKU'
    }),
    on=['month_date', 'key'],
    how='left'
)   
assert len_before_merge == len(final_df)
del len_before_merge

In [831]:
final_df[final_df['month_date'] == '2026-05-31']['key'].nunique()

0

In [832]:
# x = final_df.copy()
# x.rename(columns = {'material_group_code':'Brand'},inplace = True)
# len_before_merge = len(x)
# x = x.merge(
#     qtr_ind_rate_df.drop('month_date', axis=1).rename(columns={
#         'brand_code': 'Brand',
#         'qtr_ind_rate': 'Index Rate'
#     }),
#     on=['Brand'],
#     how='left'
# )
# assert len_before_merge == len(x)
# del len_before_merge
# x['offtake_val'] = x['Offtake Chain PSKU']*x['Index Rate']/10**7
# x.groupby(['month_date'])['offtake_val'].sum().reset_index()#[20:]

In [833]:
# lits = []
# for k in offtakes_monthly_df[offtakes_monthly_df['month_date'] == '2026-05-31']['key'].unique():
#     if(k not in final_df['key'].unique()):
#         lits.append(k)

In [834]:
# lits

In [835]:
# len(lits)

In [836]:
# final_df[final_df['Offtake Chain PSKU'].isna()]

In [837]:
mappings = {}

for run_month in final_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   

{Timestamp('2026-08-31 00:00:00'): {Timestamp('2026-08-31 00:00:00'): 'M',
  Timestamp('2026-09-30 00:00:00'): 'M+1',
  Timestamp('2026-10-31 00:00:00'): 'M+2',
  Timestamp('2026-11-30 00:00:00'): 'M+3',
  Timestamp('2026-12-31 00:00:00'): 'M+4',
  Timestamp('2027-01-31 00:00:00'): 'M+5',
  Timestamp('2027-02-28 00:00:00'): 'M+6',
  Timestamp('2027-03-31 00:00:00'): 'M+7',
  Timestamp('2027-04-30 00:00:00'): 'M+8'}}

In [838]:
final_df['M month'] = final_df.apply(
    lambda x: mappings[x['run_month']].get(x['month_date'], np.nan),
    axis=1
)

In [839]:
final_df[['run_month', 'month_date', 'M month']].drop_duplicates()

,run_month,month_date,M month
0,2026-08-31,2026-07-31,NaN
1,2026-08-31,2026-08-31,M
2,2026-08-31,2026-09-30,M+1
3,2026-08-31,2026-10-31,M+2
4,2026-08-31,2026-11-30,M+3
5,2026-08-31,2026-12-31,M+4
6,2026-08-31,2027-01-31,M+5
7,2026-08-31,2027-02-28,M+6
8,2026-08-31,2027-03-31,M+7
9,2026-08-31,2027-04-30,M+8


### Merge Forecasts

In [840]:
forecasts.head()

,key,month_date,platform_name,parent_material_code,brand_code,run_month,M month,Final Heuristic Prophet Vol,skipped
0,Amazon ARIPL_718288,2026-08-31,Amazon ARIPL,718288,SAFF GOLD,2026-08-31,M,33.954,1
1,Flipkart National_722208,2026-09-30,Flipkart National,722208,SAFF GOLD,2026-08-31,M+1,0.000,0
2,Amazon ARIPL_718288,2026-10-31,Amazon ARIPL,718288,SAFF GOLD,2026-08-31,M+2,33.954,1
3,Amazon ARIPL_718288,2026-11-30,Amazon ARIPL,718288,SAFF GOLD,2026-08-31,M+3,33.954,1
4,Amazon ARIPL_718288,2026-12-31,Amazon ARIPL,718288,SAFF GOLD,2026-08-31,M+4,33.954,1


In [841]:
forecasts.duplicated(subset=['key', 'month_date', 'run_month']).sum()

0

In [842]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    forecasts[['key', 'month_date', 'run_month', 'Final Heuristic Prophet Vol']],
    on=['key', 'month_date', 'run_month'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [843]:
final_df.rename(columns={'Final Heuristic Prophet Vol': 'Offtake Chain PSKU Forecast Vol'}, inplace=True)

### Norms

In [844]:
soh_df['as_on_date'].min()

Timestamp('2024-12-03 00:00:00')

In [845]:
soh_df[~soh_df['chain'].isin(top_chains)]['chain'].unique()

array(['Blinkit', 'Flipkart-Grocery', 'Purplle', 'Swiggy', 'Zepto'],
      dtype=object)

In [846]:
soh_df[soh_df['key'] == 'Amazon ARIPL_718288']

,chain,parent_material_code,as_on_date,current_soh,key,run_month
0,Amazon ARIPL,718288,2024-12-14,16.794,Amazon ARIPL_718288,2024-12-31
1,Amazon ARIPL,718288,2025-01-25,28.038,Amazon ARIPL_718288,2025-01-31
2,Amazon ARIPL,718288,2025-02-22,23.850,Amazon ARIPL_718288,2025-02-28
3,Amazon ARIPL,718288,2025-03-29,9.204,Amazon ARIPL_718288,2025-03-31
4,Amazon ARIPL,718288,2025-04-19,7.044,Amazon ARIPL_718288,2025-04-30
5,Amazon ARIPL,718288,2025-05-31,12.048,Amazon ARIPL_718288,2025-05-31
6,Amazon ARIPL,718288,2025-06-19,9.138,Amazon ARIPL_718288,2025-06-30
7,Amazon ARIPL,718288,2025-07-31,6.840,Amazon ARIPL_718288,2025-07-31
8,Amazon ARIPL,718288,2025-08-02,17.118,Amazon ARIPL_718288,2025-08-31
9,Amazon ARIPL,718288,2025-10-05,18.864,Amazon ARIPL_718288,2025-10-31


In [847]:
avg_inventory = soh_df[
    (soh_df['chain'].isin(top_chains))
].groupby(
    ['key'], as_index=False
)['current_soh'].mean()

In [848]:
soh_df['as_on_date'].min(), soh_df['as_on_date'].max()

(Timestamp('2024-12-03 00:00:00'), Timestamp('2026-07-31 00:00:00'))

In [849]:
avg_daily_offtakes = offtakes_monthly_df[
    (offtakes_monthly_df['month_date'] > '2024-11-30') &
    (offtakes_monthly_df['month_date'] < '2026-08-01')
].copy()
avg_daily_offtakes['days'] = avg_daily_offtakes['month_date'].dt.day
avg_daily_offtakes = avg_daily_offtakes.groupby(
    ['key'], as_index=False
)[['offtake_vol_rum', 'days']].sum()

In [850]:
avg_daily_offtakes

,key,offtake_vol_rum,days
0,Amazon ARIPL_718288,33.9540,31
1,Amazon ARIPL_718322,15.1300,31
2,Amazon ARIPL_718328,8.4060,31
3,Amazon ARIPL_718330,6.1650,31
4,Amazon ARIPL_718341,17.6810,31
...,...,...,...
3531,Purplle_807725,0.0000,608
3532,Purplle_809042,1.8000,396
3533,Purplle_809250,0.0223,608
3534,Purplle_810673,0.5740,181


In [851]:
avg_daily_offtakes['avg_offtakes_vol_rum'] = avg_daily_offtakes['offtake_vol_rum'] / avg_daily_offtakes['days']

In [852]:
norm_days = avg_inventory.merge(
    avg_daily_offtakes[['key', 'avg_offtakes_vol_rum']],
    on=['key'],
    how='left'
) 
assert len(norm_days) == len(avg_inventory)

In [853]:
norm_days

,key,current_soh,avg_offtakes_vol_rum
0,Amazon ARIPL_718288,21.238200,1.095290
1,Amazon ARIPL_718312,0.055778,NaN
2,Amazon ARIPL_718322,11.267250,0.488065
3,Amazon ARIPL_718328,4.171900,0.271161
4,Amazon ARIPL_718330,3.980500,0.198871
...,...,...,...
2663,Nykaa_810673,1.972250,0.059221
2664,Nykaa_810674,1.050000,0.017182
2665,Nykaa_811005,1.512000,NaN
2666,Nykaa_811019,40.098000,0.493175


In [854]:
norm_days['norm_days'] = norm_days['current_soh'] / norm_days['avg_offtakes_vol_rum']

In [855]:
# norm_days['norm_days'] = np.minimum(np.maximum(norm_days['norm_days'] - 30, 0), 30)

norm_days['norm_days'] = np.maximum(np.minimum(norm_days['norm_days'], 30), 5)

In [856]:
norm_days['norm_days'] = norm_days['norm_days'].fillna(5)

In [857]:
norms = final_df.copy()

In [858]:
norms = norms[['key', 'run_month', 'month_date', 'Offtake Chain PSKU Forecast Vol']]

In [859]:
norms['total_days_in_month'] = norms['month_date'].dt.day

In [860]:
norms.isnull().sum()

key                                    0
run_month                              0
month_date                             0
Offtake Chain PSKU Forecast Vol    36312
total_days_in_month                    0
dtype: int64

In [861]:
norms

,key,run_month,month_date,Offtake Chain PSKU Forecast Vol,total_days_in_month
0,Amazon ARIPL_715098,2026-08-31,2026-07-31,NaN,31
1,Amazon ARIPL_715098,2026-08-31,2026-08-31,NaN,31
2,Amazon ARIPL_715098,2026-08-31,2026-09-30,NaN,30
3,Amazon ARIPL_715098,2026-08-31,2026-10-31,NaN,31
4,Amazon ARIPL_715098,2026-08-31,2026-11-30,NaN,30
...,...,...,...,...,...
60915,Nykaa_811564,2026-08-31,2026-12-31,NaN,31
60916,Nykaa_811564,2026-08-31,2027-01-31,NaN,31
60917,Nykaa_811564,2026-08-31,2027-02-28,NaN,28
60918,Nykaa_811564,2026-08-31,2027-03-31,NaN,31


In [862]:
norms.isnull().sum()

key                                    0
run_month                              0
month_date                             0
Offtake Chain PSKU Forecast Vol    36312
total_days_in_month                    0
dtype: int64

In [863]:
len_before_merge = len(norms)
norms = norms.merge(
    norm_days[['key', 'norm_days']],
    on=['key'],
    how='left'
)
assert len_before_merge == len(norms)
del len_before_merge

In [864]:
norms['safety_stock'] = norms['Offtake Chain PSKU Forecast Vol'] *  norms['norm_days'] / norms['total_days_in_month']

In [865]:
norm_days[norm_days.duplicated(subset=['key'])]

,key,current_soh,avg_offtakes_vol_rum,norm_days


In [866]:
final_df[final_df['key'] == 'Amazon ARIPL_715098']

,key,chain,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,Offtake Chain PSKU,M month,Offtake Chain PSKU Forecast Vol
0,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-07-31,0.0,0.0,0.0,NaN,NaN,NaN
1,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-08-31,0.0,0.0,0.0,NaN,M,NaN
2,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-09-30,0.0,0.0,0.0,NaN,M+1,NaN
3,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-10-31,0.0,0.0,0.0,NaN,M+2,NaN
4,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-11-30,0.0,0.0,0.0,NaN,M+3,NaN
5,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-12-31,0.0,0.0,0.0,NaN,M+4,NaN
6,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2027-01-31,NaN,NaN,0.0,NaN,M+5,NaN
7,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2027-02-28,NaN,NaN,0.0,NaN,M+6,NaN
8,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2027-03-31,NaN,NaN,0.0,NaN,M+7,NaN
9,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2027-04-30,NaN,NaN,0.0,NaN,M+8,NaN


In [867]:
final_df[final_df.duplicated(subset=['key', 'run_month', 'month_date'])]#['material_group_code'].unique()

,key,chain,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,Offtake Chain PSKU,M month,Offtake Chain PSKU Forecast Vol


In [868]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    norms.drop(['Offtake Chain PSKU Forecast Vol', 'total_days_in_month', 'norm_days'], axis=1).rename(columns={
        'safety_stock': 'norms_soh'
    }),
    on=['key', 'run_month', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [869]:
norms['month_date'] = norms['month_date'] - MonthEnd(1)

In [870]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    norms.drop(['Offtake Chain PSKU Forecast Vol', 'total_days_in_month'], axis=1),
    on=['key', 'run_month', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [871]:
final_df.sort_values(
    by=['run_month', 'material_group_code', 'key', 'month_date'], inplace=True
)

In [872]:
final_df['safety_stock'] = final_df['safety_stock'].fillna(0)

In [873]:
chain_wise_max_soh_dates = soh_df[soh_df['chain'].isin(top_chains)].groupby(
    ['run_month', 'chain'], as_index=False
)['as_on_date'].max()

chain_wise_max_soh_dates

,run_month,chain,as_on_date
0,2024-12-31,Amazon ARIPL,2024-12-14
1,2024-12-31,Amazon RK,2024-12-19
2,2024-12-31,Big Basket,2024-12-21
3,2024-12-31,Flipkart Grocery,2024-12-16
4,2024-12-31,Flipkart National,2024-12-20
...,...,...,...
124,2026-07-31,Flipkart Grocery,2026-07-31
125,2026-07-31,Flipkart National,2026-07-31
126,2026-07-31,Meesho,2026-07-31
127,2026-07-31,Myntra,2026-07-31


In [874]:
chain_wise_max_soh_dates = (
    chain_wise_max_soh_dates
    .groupby('chain')
    .apply(lambda x: dict(zip(x['run_month'], x['as_on_date'])))
    .to_dict()
)

In [875]:
last_date_soh = pd.DataFrame()

for c in chain_wise_max_soh_dates.keys():
    for rm in chain_wise_max_soh_dates[c].keys():
        last_date_soh = pd.concat([
            last_date_soh,
            soh_df[
                (soh_df['chain'] == c) &
                (soh_df['as_on_date'] == chain_wise_max_soh_dates[c][rm])
            ]
        ])

In [876]:
last_date_soh['as_on_date'] = last_date_soh['as_on_date'] + MonthEnd(0)

In [877]:
last_date_soh.groupby(['as_on_date'])['current_soh'].sum()

as_on_date
2024-12-31    246778.066431
2025-01-31    311218.299326
2025-02-28    192679.204553
2025-03-31    244457.303018
2025-04-30    366643.182600
2025-05-31    319795.321935
2025-06-30    398319.051492
2025-07-31    114248.954225
2025-08-31    698712.990596
2025-09-30    774083.078400
2025-10-31    397742.700376
2025-11-30    251355.181461
2025-12-31    435443.790881
2026-01-31    483352.629753
2026-02-28     69373.308492
2026-03-31    282498.423831
2026-04-30    294009.295182
2026-05-31    405360.322214
2026-06-30    428900.652800
2026-07-31    727605.579907
Name: current_soh, dtype: float64

In [878]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    last_date_soh[['key', 'as_on_date', 'current_soh']].rename(
        columns={
            'as_on_date': 'month_date',
            'current_soh': 'Actual Closing SOH'
        }
    ),
    on=['key', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)

del len_before_merge

In [879]:
final_df.sort_values(by=['run_month', 'key', 'month_date'], inplace=True)

In [880]:
final_df['Actual Closing SOH_Lag_1'] = final_df.groupby(
    ['run_month', 'key']
)['Actual Closing SOH'].shift(1)

final_df['Actual Closing SOH Lag 2'] = final_df.groupby(
    ['run_month', 'key']
)['Actual Closing SOH'].shift(2)

In [881]:
final_df['safety_stock'].sum()

3627319.152435735

In [882]:
final_df['safety_stock'].min()

0.0

In [883]:
# final_df['Assumed Closing SOH'] = np.where(
#     final_df['M month'] == 'M',  
#     final_df['Actual Closing SOH_Lag_1'].fillna(0) + final_df['sec_apo_plan_vol_rum'].fillna(0) \
#     - final_df['Offtake Chain PSKU Forecast Vol'].fillna(0),
#     final_df['safety_stock']
# )

final_df['Assumed Closing SOH'] = final_df['safety_stock']

In [884]:
final_df['Assumed Closing SOH'] = final_df['Assumed Closing SOH'].clip(lower=0.0)

In [885]:
final_df['Assumed Closing SOH_Lag_1'] = final_df.groupby(
    ['run_month', 'key']
)['Assumed Closing SOH'].shift(1)

final_df['Assumed Closing SOH Lag 2'] = final_df.groupby(
    ['run_month', 'key']
)['Assumed Closing SOH'].shift(2)

In [886]:
final_df = final_df[
    ~final_df['material_group_code'].isin(['NC FREE', 'HC FREE'])
]

In [887]:
final_df.isna().sum()

key                                    0
chain                                  0
parent_material_code                   0
material_group_code                    0
run_month                              0
month_date                             0
pri_actuals_vol_rum                24389
sec_apo_plan_vol_rum               24389
Primary P3M                           21
Offtake Chain PSKU                 30850
M month                             6092
Offtake Chain PSKU Forecast Vol    36312
norms_soh                          42704
norm_days                          39752
safety_stock                           0
Actual Closing SOH                 59339
Actual Closing SOH_Lag_1           59339
Actual Closing SOH Lag 2           59339
Assumed Closing SOH                    0
Assumed Closing SOH_Lag_1           6092
Assumed Closing SOH Lag 2          12184
dtype: int64

In [888]:
final_df.shape

(60920, 21)

### Add P3M, LY

In [889]:
actuals_df = plan_actuals_df.copy()
actuals_df['month_date'] = actuals_df['month_date'] + MonthEnd(0)

In [890]:
actuals_df = actuals_df.groupby(
    ['key', 'month_date'], as_index=False
)[['pri_actuals_vol_rum', 'sec_actuals_vol_rum']].sum()

In [891]:
actuals_df.duplicated(subset=['key', 'month_date']).sum()

0

In [892]:
actuals_df['month_date'].min()

Timestamp('2023-01-31 00:00:00')

In [893]:
actuals_df.sort_values(by=['key', 'month_date'], inplace=True)

In [894]:
actuals_df['Primary P3M redundant'] = actuals_df.groupby(
['key'], as_index = False, group_keys = False)['pri_actuals_vol_rum'].shift(1)\
                            .rolling(window=3, min_periods=1).mean()

In [895]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    actuals_df.rename(columns={
        'pri_actuals_vol_rum': 'Primary Actuals Vol',
        'sec_actuals_vol_rum': 'Sec Actuals Vol'
    }),
    on=['key', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [896]:
ly_actuals_df = actuals_df.copy()

In [897]:
ly_actuals_df['month_date'] = ly_actuals_df['month_date'] + MonthEnd(12)

In [898]:
ly_actuals_df.rename(columns={
        'pri_actuals_vol_rum': 'LY Primary Actuals Vol',
        'sec_actuals_vol_rum': 'LY Sec Actuals Vol',
        'Primary P3M redundant': 'LY Primary P3M'
    })

,key,month_date,LY Primary Actuals Vol,LY Sec Actuals Vol,LY Primary P3M
0,Amazon ARIPL_715098,2024-07-31,0.0,0.0,NaN
1,Amazon ARIPL_715098,2024-08-31,0.0,0.0,0.0
2,Amazon ARIPL_715098,2024-09-30,0.0,0.0,0.0
3,Amazon ARIPL_715098,2024-10-31,0.0,0.0,0.0
4,Amazon ARIPL_715098,2024-11-30,0.0,0.0,0.0
...,...,...,...,...,...
218487,Nykaa_811564,2027-08-31,0.0,0.0,0.0
218488,Nykaa_811564,2027-09-30,0.0,0.0,0.0
218489,Nykaa_811564,2027-10-31,0.0,0.0,0.0
218490,Nykaa_811564,2027-11-30,0.0,0.0,0.0


In [899]:
ly_actuals_df.sort_values(by=['key', 'month_date'], inplace=True)

In [900]:
ly_actuals_df['pri_actuals_vol_rum_lag_1'] = ly_actuals_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(1)

ly_actuals_df['pri_actuals_vol_rum_lag_2'] = ly_actuals_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(2)

ly_actuals_df['pri_actuals_vol_rum_lag_3'] = ly_actuals_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(3)


ly_actuals_df['pri_actuals_vol_rum_lead_1'] = ly_actuals_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(-1)

ly_actuals_df['pri_actuals_vol_rum_lead_2'] = ly_actuals_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(-2)

In [901]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    ly_actuals_df.rename(columns={
        'pri_actuals_vol_rum': 'LY Primary Actuals Vol',
        'sec_actuals_vol_rum': 'LY Sec Actuals Vol',
        'Primary P3M redundant': 'LY Primary P3M', 
        'pri_actuals_vol_rum_lag_1': 'LY Primary Actuals Lag 1 Vol',
        'pri_actuals_vol_rum_lag_2': 'LY Primary Actuals Lag 2 Vol',
        'pri_actuals_vol_rum_lag_3': 'LY Primary Actuals Lag 3 Vol',
        'pri_actuals_vol_rum_lead_1': 'LY Primary Actuals Lead 1 Vol',
        'pri_actuals_vol_rum_lead_2': 'LY Primary Actuals Lead 2 Vol',
    }),
    on=['key', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [902]:
prev_offtakes_monthly_df = offtakes_monthly_df.copy()

In [903]:
final_offtakes_historical_df = prev_offtakes_monthly_df.copy()

In [904]:
final_offtakes_historical_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()

0

In [905]:
final_offtakes_historical_df['key'] = final_offtakes_historical_df[['chain', 'parent_material_code']].astype(str).agg(
    '_'.join, axis=1
)

In [906]:

final_offtakes_historical_df.sort_values(
    by=['key', 'month_date'], inplace=True
)

In [907]:
final_offtakes_historical_df['P3M'] = final_offtakes_historical_df.groupby(
    ['key'], as_index = False, group_keys = False)['offtake_vol_rum'].shift(1)\
                                .rolling(window=3, min_periods=3).mean()

In [908]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    final_offtakes_historical_df[['key', 'month_date', 'offtake_vol_rum', 'P3M']].rename(
        columns={'P3M': 'Offtake P3M', 'offtake_vol_rum': 'Offtake Actuals Vol'}
    ),
    on=['key', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [909]:
ly_final_offtakes_historical_df = final_offtakes_historical_df.copy()

In [910]:

ly_final_offtakes_historical_df.sort_values(
    by=['key', 'month_date'], inplace=True
)

In [911]:
ly_final_offtakes_historical_df['month_date'].min()

Timestamp('2023-01-31 00:00:00')

In [912]:
ly_final_offtakes_historical_df['month_date'] = ly_final_offtakes_historical_df['month_date'] + MonthEnd(12)
ly_final_offtakes_historical_df.head()

,chain,parent_material_code,material_group_code,month_date,offtake_vol_rum,portfolio,key,P3M
0,Amazon ARIPL,718288,SAFF GOLD,2027-07-31,33.954,SAFFOLA OILS,Amazon ARIPL_718288,NaN
1,Amazon ARIPL,718288,SAFF GOLD,2027-08-31,0.000,SAFFOLA OILS,Amazon ARIPL_718288,NaN
2,Amazon ARIPL,718288,SAFF GOLD,2027-09-30,0.000,SAFFOLA OILS,Amazon ARIPL_718288,NaN
3,Amazon ARIPL,718288,SAFF GOLD,2027-10-31,0.000,SAFFOLA OILS,Amazon ARIPL_718288,11.318
4,Amazon ARIPL,718288,SAFF GOLD,2027-11-30,0.000,SAFFOLA OILS,Amazon ARIPL_718288,0.000


In [913]:
ly_final_offtakes_historical_df['LY Offtake Actuals Lag 1 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(1)

ly_final_offtakes_historical_df['LY Offtake Actuals Lag 2 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(2)

ly_final_offtakes_historical_df['LY Offtake Actuals Lag 3 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(3)


In [914]:
ly_final_offtakes_historical_df['LY Offtake Actuals Lead 1 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(-1)

ly_final_offtakes_historical_df['LY Offtake Actuals Lead 2 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(-2)

In [915]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    ly_final_offtakes_historical_df[['key', 'month_date', 'offtake_vol_rum', 'P3M', 
                                     'LY Offtake Actuals Lag 1 Vol', 'LY Offtake Actuals Lag 2 Vol', 
                                     'LY Offtake Actuals Lag 3 Vol', 'LY Offtake Actuals Lead 1 Vol',
                                     'LY Offtake Actuals Lead 2 Vol']].rename(
        columns={'P3M': 'LY Offtake P3M', 'offtake_vol_rum': 'LY Offtake Actuals Vol'}
    ),
    on=['key', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [916]:
final_df.sort_values(by=['run_month', 'key', 'month_date'], inplace=True)

In [917]:
for col in ['Primary P3M', 'LY Primary P3M', 'Offtake P3M', 'LY Offtake P3M', 'Primary P3M redundant']:
    # if not 'LY' in col:  'LY P6M',
    final_df.loc[final_df['month_date'] > final_df['run_month'], [col]] = np.nan
    final_df[col] = final_df.groupby(['run_month', 'key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [918]:
final_offtakes_historical_df.sort_values(by=['key', 'month_date'], inplace=True)

In [919]:
lags_df = plan_actuals_df.copy()

In [920]:
lags_df

,month_date,key,chain,parent_material_code,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,Primary P3M
24455,2023-07-31,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.00000,8.3240,0.0,0.0
24456,2023-08-31,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.00000,8.3240,0.0,0.0
24457,2023-09-30,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.00000,0.0000,0.0,0.0
24458,2023-10-31,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.00000,0.0000,0.0,0.0
24459,2023-11-30,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.00000,0.0000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
339843,2026-08-31,Nykaa_811564,Nykaa,811564,SAF-MUSLI,0.0,0.65975,0.4953,0.0,0.0
339844,2026-09-30,Nykaa_811564,Nykaa,811564,SAF-MUSLI,0.0,0.00000,0.0000,0.0,0.0
339845,2026-10-31,Nykaa_811564,Nykaa,811564,SAF-MUSLI,0.0,0.00000,0.0000,0.0,0.0
339846,2026-11-30,Nykaa_811564,Nykaa,811564,SAF-MUSLI,0.0,0.00000,0.0000,0.0,0.0


In [921]:
lags_df.sort_values(by=['key', 'month_date'], inplace=True)

In [922]:
lags_df

,month_date,key,chain,parent_material_code,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,Primary P3M
24455,2023-07-31,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.00000,8.3240,0.0,0.0
24456,2023-08-31,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.00000,8.3240,0.0,0.0
24457,2023-09-30,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.00000,0.0000,0.0,0.0
24458,2023-10-31,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.00000,0.0000,0.0,0.0
24459,2023-11-30,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.00000,0.0000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
339843,2026-08-31,Nykaa_811564,Nykaa,811564,SAF-MUSLI,0.0,0.65975,0.4953,0.0,0.0
339844,2026-09-30,Nykaa_811564,Nykaa,811564,SAF-MUSLI,0.0,0.00000,0.0000,0.0,0.0
339845,2026-10-31,Nykaa_811564,Nykaa,811564,SAF-MUSLI,0.0,0.00000,0.0000,0.0,0.0
339846,2026-11-30,Nykaa_811564,Nykaa,811564,SAF-MUSLI,0.0,0.00000,0.0000,0.0,0.0


In [923]:
lags_df['Primary_Lag_2'] = lags_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(1)

lags_df['Primary_Lag_3'] = lags_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(2)

In [924]:
lags_df['month_date'] = lags_df['month_date'] + MonthEnd(1)

In [925]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    lags_df[['key', 'month_date', 'pri_actuals_vol_rum', 'Primary_Lag_2', 'Primary_Lag_3']].rename(columns={
        'month_date': 'run_month',
        'pri_actuals_vol_rum': 'Primary_Lag_1'
    }),
    on=['key', 'run_month'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [926]:
lags_final_offtakes_historical_df = final_offtakes_historical_df.copy()

lags_final_offtakes_historical_df.sort_values(by=['key', 'month_date'], inplace=True)
lags_final_offtakes_historical_df['OT_Lag_2'] = lags_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(1)

lags_final_offtakes_historical_df['OT_Lag_3'] = lags_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(2)

In [927]:
# x = lags_final_offtakes_historical_df.copy()
# x.rename(columns = {'material_group_code':'Brand'},inplace = True)
# len_before_merge = len(x)
# x = x.merge(
#     qtr_ind_rate_df.drop('month_date', axis=1).rename(columns={
#         'brand_code': 'Brand',
#         'qtr_ind_rate': 'Index Rate'
#     }),
#     on=['Brand'],
#     how='left'
# )
# assert len_before_merge == len(x)
# del len_before_merge
# x['offtake_val_lag3'] = x['OT_Lag_3']*x['Index Rate']/10**7
# x['offtake_val_lag2'] = x['OT_Lag_2']*x['Index Rate']/10**7
# x['offtake_val_lag1'] = x['offtake_vol_rum']*x['Index Rate']/10**7
# x

In [928]:
# x.columns
# x[x['month_date'] == '2026-06-30'][['offtake_val_lag3', 'offtake_val_lag2',
#        'offtake_val_lag1']].sum()

In [929]:
# x[(x['key'].isin(final_df['key'].unique())) & (x['month_date'] == '2026-06-30')][['offtake_val_lag3', 'offtake_val_lag2',
#        'offtake_val_lag1']].sum()

In [930]:
# x[(~x['key'].isin(final_df['key'].unique())) & (x['month_date'] == '2026-06-30')].to_csv('key_check.csv')

In [931]:
lags_final_offtakes_historical_df

,chain,parent_material_code,material_group_code,month_date,offtake_vol_rum,portfolio,key,P3M,OT_Lag_2,OT_Lag_3
0,Amazon ARIPL,718288,SAFF GOLD,2026-07-31,33.954,SAFFOLA OILS,Amazon ARIPL_718288,NaN,NaN,NaN
1,Amazon ARIPL,718288,SAFF GOLD,2026-08-31,0.000,SAFFOLA OILS,Amazon ARIPL_718288,NaN,33.954,NaN
2,Amazon ARIPL,718288,SAFF GOLD,2026-09-30,0.000,SAFFOLA OILS,Amazon ARIPL_718288,NaN,0.000,33.954
3,Amazon ARIPL,718288,SAFF GOLD,2026-10-31,0.000,SAFFOLA OILS,Amazon ARIPL_718288,11.318,0.000,0.000
4,Amazon ARIPL,718288,SAFF GOLD,2026-11-30,0.000,SAFFOLA OILS,Amazon ARIPL_718288,0.000,0.000,0.000
...,...,...,...,...,...,...,...,...,...,...
138925,Purplle,810674,PA_ESS_HO,2027-01-31,0.000,HAIR OILS,Purplle_810674,0.000,0.000,0.000
138926,Purplle,810674,PA_ESS_HO,2027-02-28,0.000,HAIR OILS,Purplle_810674,0.000,0.000,0.000
138927,Purplle,810674,PA_ESS_HO,2027-03-31,0.000,HAIR OILS,Purplle_810674,0.000,0.000,0.000
138928,Purplle,810674,PA_ESS_HO,2027-04-30,0.000,HAIR OILS,Purplle_810674,0.000,0.000,0.000


In [932]:
lags_final_offtakes_historical_df['month_date'] = lags_final_offtakes_historical_df['month_date'] + MonthEnd(1)

In [933]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    lags_final_offtakes_historical_df[['key', 'month_date', 'offtake_vol_rum', 'OT_Lag_2', 'OT_Lag_3']].rename(columns={
        'month_date': 'run_month',
        'offtake_vol_rum': 'OT_Lag_1'
    }),
    on=['key', 'run_month'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [934]:
final_df

,key,chain,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,Offtake Chain PSKU,...,LY Offtake Actuals Lag 2 Vol,LY Offtake Actuals Lag 3 Vol,LY Offtake Actuals Lead 1 Vol,LY Offtake Actuals Lead 2 Vol,Primary_Lag_1,Primary_Lag_2,Primary_Lag_3,OT_Lag_1,OT_Lag_2,OT_Lag_3
0,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-07-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
1,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-08-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
2,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-09-30,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
3,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-10-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
4,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-11-30,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60915,Nykaa_811564,Nykaa,811564,SAF-MUSLI,2026-08-31,2026-12-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
60916,Nykaa_811564,Nykaa,811564,SAF-MUSLI,2026-08-31,2027-01-31,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
60917,Nykaa_811564,Nykaa,811564,SAF-MUSLI,2026-08-31,2027-02-28,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
60918,Nykaa_811564,Nykaa,811564,SAF-MUSLI,2026-08-31,2027-03-31,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [935]:
# x = final_df.copy()
# x.rename(columns = {'material_group_code':'Brand'},inplace = True)
# len_before_merge = len(x)
# x = x.merge(
#     qtr_ind_rate_df.drop('month_date', axis=1).rename(columns={
#         'brand_code': 'Brand',
#         'qtr_ind_rate': 'Index Rate'
#     }),
#     on=['Brand'],
#     how='left'
# )
# assert len_before_merge == len(x)
# del len_before_merge
# x['offtake_val'] = x['OT_Lag_2']*x['Index Rate']/10**7
# x.groupby(['month_date'])['offtake_val'].sum().reset_index()#[20:]

In [936]:
final_df

,key,chain,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,Offtake Chain PSKU,...,LY Offtake Actuals Lag 2 Vol,LY Offtake Actuals Lag 3 Vol,LY Offtake Actuals Lead 1 Vol,LY Offtake Actuals Lead 2 Vol,Primary_Lag_1,Primary_Lag_2,Primary_Lag_3,OT_Lag_1,OT_Lag_2,OT_Lag_3
0,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-07-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
1,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-08-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
2,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-09-30,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
3,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-10-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
4,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-11-30,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60915,Nykaa_811564,Nykaa,811564,SAF-MUSLI,2026-08-31,2026-12-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
60916,Nykaa_811564,Nykaa,811564,SAF-MUSLI,2026-08-31,2027-01-31,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
60917,Nykaa_811564,Nykaa,811564,SAF-MUSLI,2026-08-31,2027-02-28,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
60918,Nykaa_811564,Nykaa,811564,SAF-MUSLI,2026-08-31,2027-03-31,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [937]:
# final_df.to_csv('ECOM_OTP_v0_check.csv', index=False)

In [938]:
forecasts['run_month'].unique()

<DatetimeArray>
['2026-08-31 00:00:00']
Length: 1, dtype: datetime64[ns]

In [939]:
def calculate_primary_iteratively(key_df):
    key_df = key_df.sort_values('month_date').copy()
    key_df = key_df[key_df['month_date'] >= key_df['run_month']]

    # Fill NaNs
    fill_cols = [
        'Offtake Chain PSKU Forecast Vol',
        'safety_stock',
        'Actual Closing SOH_Lag_1'
    ]
    key_df[fill_cols] = key_df[fill_cols].fillna(0)

    _key = key_df['key'].iloc[0]
    _run_month = key_df['run_month'].iloc[0]

    outputs = []
    prev_soh = None

    for _, row in key_df.iterrows():
        forecast = row['Offtake Chain PSKU Forecast Vol']
        safety_stock = row['safety_stock']

        if row['M month'] == 'M':
            opening_soh = row['Actual Closing SOH_Lag_1']
        else:
            opening_soh = prev_soh

        primary_vol = max(
            forecast + safety_stock - opening_soh,
            0
        )

        assumed_closing_soh = max(
            opening_soh + primary_vol - forecast,
            0
        )

        outputs.append({
            'run_month': _run_month,
            'key': _key,
            'month_date': row['month_date'],
            'Calculated Primary Vol': primary_vol,
            'Final Assumed Closing SOH Vol': assumed_closing_soh
        })

        prev_soh = assumed_closing_soh

    return outputs

In [940]:
# # final_df['Calculated Primary Vol'] = np.where(
# #     final_df['M month'] == 'M',
# #     final_df['sec_apo_plan_vol_rum'],
# #     final_df['Offtake Chain PSKU Forecast Vol'].fillna(0) + \
# #         final_df['safety_stock'].fillna(0) - \
# #         final_df['Assumed Closing SOH_Lag_1'].fillna(0)
# # )


# final_df['Calculated Primary Vol'] = final_df['Offtake Chain PSKU Forecast Vol'].fillna(0) + \
#     final_df['safety_stock'].fillna(0) - final_df['Assumed Closing SOH_Lag_1'].fillna(0)

In [941]:
# final_df['Calculated Primary Vol'] = final_df['Calculated Primary Vol'].clip(lower=0)

In [942]:
final_df

,key,chain,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,Offtake Chain PSKU,...,LY Offtake Actuals Lag 2 Vol,LY Offtake Actuals Lag 3 Vol,LY Offtake Actuals Lead 1 Vol,LY Offtake Actuals Lead 2 Vol,Primary_Lag_1,Primary_Lag_2,Primary_Lag_3,OT_Lag_1,OT_Lag_2,OT_Lag_3
0,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-07-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
1,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-08-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
2,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-09-30,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
3,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-10-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
4,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-11-30,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60915,Nykaa_811564,Nykaa,811564,SAF-MUSLI,2026-08-31,2026-12-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
60916,Nykaa_811564,Nykaa,811564,SAF-MUSLI,2026-08-31,2027-01-31,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
60917,Nykaa_811564,Nykaa,811564,SAF-MUSLI,2026-08-31,2027-02-28,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
60918,Nykaa_811564,Nykaa,811564,SAF-MUSLI,2026-08-31,2027-03-31,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [943]:
calculated_primary = []

for (_, _), group_df in tqdm(final_df.groupby(['run_month', 'key'])):
    calculated_primary.extend(
        calculate_primary_iteratively(group_df)
    )

calculated_primary_df = pd.DataFrame(calculated_primary)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 6092/6092 [00:14<00:00, 435.12it/s]


In [944]:
del calculated_primary

In [945]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    calculated_primary_df,
    on=['run_month', 'month_date', 'key'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [946]:
final_df['Calculated Primary Vol'].min(), final_df['Final Assumed Closing SOH Vol'].min()

(0.0, 0.0)

In [947]:
final_df.sort_values(by=['run_month', 'key', 'month_date'], inplace=True)

In [948]:
final_df['Final Assumed Closing SOH Lag 1 Vol'] = final_df.groupby(
    ['run_month', 'key']
)['Final Assumed Closing SOH Vol'].shift(1)

final_df['Final Assumed Closing SOH Lag 2 Vol'] = final_df.groupby(
    ['run_month', 'key']
)['Final Assumed Closing SOH Vol'].shift(2)

In [949]:
final_df

,key,chain,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,Offtake Chain PSKU,...,Primary_Lag_1,Primary_Lag_2,Primary_Lag_3,OT_Lag_1,OT_Lag_2,OT_Lag_3,Calculated Primary Vol,Final Assumed Closing SOH Vol,Final Assumed Closing SOH Lag 1 Vol,Final Assumed Closing SOH Lag 2 Vol
0,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-07-31,0.0,0.0,0.0,NaN,...,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-08-31,0.0,0.0,0.0,NaN,...,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN
2,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-09-30,0.0,0.0,0.0,NaN,...,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.0,0.0,NaN
3,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-10-31,0.0,0.0,0.0,NaN,...,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0
4,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-08-31,2026-11-30,0.0,0.0,0.0,NaN,...,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60915,Nykaa_811564,Nykaa,811564,SAF-MUSLI,2026-08-31,2026-12-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
60916,Nykaa_811564,Nykaa,811564,SAF-MUSLI,2026-08-31,2027-01-31,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
60917,Nykaa_811564,Nykaa,811564,SAF-MUSLI,2026-08-31,2027-02-28,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
60918,Nykaa_811564,Nykaa,811564,SAF-MUSLI,2026-08-31,2027-03-31,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0


In [950]:
brand_md_df

,brand_code,portfolio
0,BD_BDOL_M,BEARDO
1,BD_HRWX_M,BEARDO
2,BRD_BDCOL,BEARDO
3,BRD_BDOIL,BEARDO
4,BRD_BDSPR,BEARDO
...,...,...
315,PURSNS_GM,SKIN CARE
316,PURSNS_ML,SKIN CARE
317,TO_CL_OTG,SKIN CARE
318,TO_CL_OTM,SKIN CARE


In [951]:
def read_qtr_ind_rate_table():
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=prod_conn, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    
    return qtr_ind_rate

In [952]:
qtr_ind_rate_df = read_qtr_ind_rate_table()
qtr_ind_rate_df.head()

,month_date,brand_code,qtr_ind_rate
0,2027-03-31,CMX_WELPD,3014.000
1,2027-03-31,4700_BCPC,222.000
2,2027-03-31,CMX_PRTPD,3014.000
3,2027-03-31,PA_CN_HGO,488.152
4,2027-03-31,TRU_RAWDF,800.000


In [953]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(
        columns={'brand_code': 'material_group_code'}
    ),
    on=['material_group_code'], 
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [954]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    brand_md_df.rename(
        columns={'brand_code': 'material_group_code'}
    ),
    on=['material_group_code'], 
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [955]:
final_df.columns

Index(['key', 'chain', 'parent_material_code', 'material_group_code',
       'run_month', 'month_date', 'pri_actuals_vol_rum',
       'sec_apo_plan_vol_rum', 'Primary P3M', 'Offtake Chain PSKU', 'M month',
       'Offtake Chain PSKU Forecast Vol', 'norms_soh', 'norm_days',
       'safety_stock', 'Actual Closing SOH', 'Actual Closing SOH_Lag_1',
       'Actual Closing SOH Lag 2', 'Assumed Closing SOH',
       'Assumed Closing SOH_Lag_1', 'Assumed Closing SOH Lag 2',
       'Primary Actuals Vol', 'Sec Actuals Vol', 'Primary P3M redundant',
       'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
       'LY Primary Actuals Lead 2 Vol', 'Offtake Actuals Vol', 'Offtake P3M',
       'LY Offtake Actuals Vol', 'LY Offtake P3M',
       'LY Offtake Actuals Lag 1 Vol', 'LY Offtake Actuals Lag 2 Vol',
       'LY Offtake Actuals Lag 3 Vol', '

In [956]:
final_df = final_df.rename(columns={
    'key': 'Key',
    'chain': 'Chain',
    'parent_material_code': 'PSKU',
    'material_group_code': 'Brand',
    'run_month': 'Run Month',
    'month_date': 'Month Date',
    'pri_actuals_vol_rum': 'Primary Till Date Actuals Vol',
    'sec_apo_plan_vol_rum': 'Secondary Plan Vol',
    'Primary P3M': 'Primary P3M Vol',
    'Offtake Chain PSKU': 'Offtake Chain PSKU Vol',
    'norms_soh': 'Norms SOH',
    'norm_days': 'Norm Days',
    'safety_stock': 'Safety Stock Vol',
    'Actual Closing SOH': 'Actual Closing SOH Vol',
    'Actual Closing SOH_Lag_1': 'Actual Closing SOH Lag 1 Vol',
    'Actual Closing SOH Lag 2': 'Actual Closing SOH Lag 2 Vol',
    'Assumed Closing SOH': 'Assumed Closing SOH Vol',
    'Assumed Closing SOH_Lag_1': 'Assumed Closing SOH Lag 1 Vol',
    'Primary P3M redundant': 'Primary P3M redundant Vol',
    'LY Primary P3M': 'LY Primary P3M Vol',
    'Offtake P3M': 'Offtake P3M Vol',
    'LY Offtake P3M': 'LY Offtake P3M Vol',
    'Primary_Lag_1': 'Primary Actuals Lag 1 Vol',
    'Primary_Lag_2': 'Primary Actuals Lag 2 Vol',
    'Primary_Lag_3': 'Primary Actuals Lag 3 Vol',
    'OT_Lag_1': 'Offtake Actuals Lag 1 Vol',
    'OT_Lag_2': 'Offtake Actuals Lag 2 Vol',
    'OT_Lag_3': 'Offtake Actuals Lag 3 Vol',
    'qtr_ind_rate': 'Index Rate',
    'portfolio': 'Portfolio'
})

In [957]:
final_df.columns

Index(['Key', 'Chain', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Primary Till Date Actuals Vol', 'Secondary Plan Vol',
       'Primary P3M Vol', 'Offtake Chain PSKU Vol', 'M month',
       'Offtake Chain PSKU Forecast Vol', 'Norms SOH', 'Norm Days',
       'Safety Stock Vol', 'Actual Closing SOH Vol',
       'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol',
       'Assumed Closing SOH Vol', 'Assumed Closing SOH Lag 1 Vol',
       'Assumed Closing SOH Lag 2', 'Primary Actuals Vol', 'Sec Actuals Vol',
       'Primary P3M redundant Vol', 'LY Primary Actuals Vol',
       'LY Sec Actuals Vol', 'LY Primary P3M Vol',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
       'LY Primary Actuals Lead 2 Vol', 'Offtake Actuals Vol',
       'Offtake P3M Vol', 'LY Offtake Actuals Vol', 'LY Offtake P3M Vol',
       'LY Offtake Actuals Lag 1 Vol', 'LY Offtake Actuals Lag 2 Vol',
       

In [958]:
final_df = final_df[[
    'Key', 'Chain', 'PSKU', 'Brand', 'Index Rate',
    'Portfolio', 'Run Month', 'Month Date',  'M month',

    'Primary Till Date Actuals Vol', 'Secondary Plan Vol',
    'Primary P3M Vol', 'Offtake Chain PSKU Vol',

    'Offtake Chain PSKU Forecast Vol', 'Norms SOH', 'Norm Days',
    'Safety Stock Vol', 
    
    'Actual Closing SOH Vol', 'Actual Closing SOH Lag 1 Vol', 
    'Actual Closing SOH Lag 2 Vol', 'Assumed Closing SOH Vol',
    'Assumed Closing SOH Lag 1 Vol', 'Final Assumed Closing SOH Vol', 
    'Final Assumed Closing SOH Lag 1 Vol', 
    
    'Primary Actuals Vol', 'Sec Actuals Vol', 'Primary P3M redundant Vol',
    'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M Vol',
    'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol', 'Primary Actuals Lag 3 Vol',
    'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
    'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
    'LY Primary Actuals Lead 2 Vol',

    'Offtake Actuals Vol', 'Offtake P3M Vol', 'LY Offtake Actuals Vol',
    'LY Offtake P3M Vol', 
    'Offtake Actuals Lag 1 Vol', 'Offtake Actuals Lag 2 Vol',
    'Offtake Actuals Lag 3 Vol', 'LY Offtake Actuals Lag 1 Vol', 
    'LY Offtake Actuals Lag 2 Vol', 'LY Offtake Actuals Lag 3 Vol', 
    'LY Offtake Actuals Lead 1 Vol', 'LY Offtake Actuals Lead 2 Vol', 

    'Calculated Primary Vol'
]]

In [959]:
vol_to_val_cols = [col for col in final_df.columns if 'Vol' in col]
vol_to_val_cols

['Primary Till Date Actuals Vol',
 'Secondary Plan Vol',
 'Primary P3M Vol',
 'Offtake Chain PSKU Vol',
 'Offtake Chain PSKU Forecast Vol',
 'Safety Stock Vol',
 'Actual Closing SOH Vol',
 'Actual Closing SOH Lag 1 Vol',
 'Actual Closing SOH Lag 2 Vol',
 'Assumed Closing SOH Vol',
 'Assumed Closing SOH Lag 1 Vol',
 'Final Assumed Closing SOH Vol',
 'Final Assumed Closing SOH Lag 1 Vol',
 'Primary Actuals Vol',
 'Sec Actuals Vol',
 'Primary P3M redundant Vol',
 'LY Primary Actuals Vol',
 'LY Sec Actuals Vol',
 'LY Primary P3M Vol',
 'Primary Actuals Lag 1 Vol',
 'Primary Actuals Lag 2 Vol',
 'Primary Actuals Lag 3 Vol',
 'LY Primary Actuals Lag 1 Vol',
 'LY Primary Actuals Lag 2 Vol',
 'LY Primary Actuals Lag 3 Vol',
 'LY Primary Actuals Lead 1 Vol',
 'LY Primary Actuals Lead 2 Vol',
 'Offtake Actuals Vol',
 'Offtake P3M Vol',
 'LY Offtake Actuals Vol',
 'LY Offtake P3M Vol',
 'Offtake Actuals Lag 1 Vol',
 'Offtake Actuals Lag 2 Vol',
 'Offtake Actuals Lag 3 Vol',
 'LY Offtake Actuals L

In [960]:
for col in vol_to_val_cols:
    final_df[col[:-3] + 'Val'] = final_df[col].fillna(0) * final_df['Index Rate'] / (10 ** 7)

In [961]:
final_df

,Key,Chain,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,M month,Primary Till Date Actuals Vol,...,LY Offtake P3M Val,Offtake Actuals Lag 1 Val,Offtake Actuals Lag 2 Val,Offtake Actuals Lag 3 Val,LY Offtake Actuals Lag 1 Val,LY Offtake Actuals Lag 2 Val,LY Offtake Actuals Lag 3 Val,LY Offtake Actuals Lead 1 Val,LY Offtake Actuals Lead 2 Val,Calculated Primary Val
0,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1220.081000,SKIN CARE,2026-08-31,2026-07-31,NaN,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1220.081000,SKIN CARE,2026-08-31,2026-08-31,M,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1220.081000,SKIN CARE,2026-08-31,2026-09-30,M+1,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1220.081000,SKIN CARE,2026-08-31,2026-10-31,M+2,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1220.081000,SKIN CARE,2026-08-31,2026-11-30,M+3,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60915,Nykaa_811564,Nykaa,811564,SAF-MUSLI,315513.490535,FOODS,2026-08-31,2026-12-31,M+4,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
60916,Nykaa_811564,Nykaa,811564,SAF-MUSLI,315513.490535,FOODS,2026-08-31,2027-01-31,M+5,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
60917,Nykaa_811564,Nykaa,811564,SAF-MUSLI,315513.490535,FOODS,2026-08-31,2027-02-28,M+6,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
60918,Nykaa_811564,Nykaa,811564,SAF-MUSLI,315513.490535,FOODS,2026-08-31,2027-03-31,M+7,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [962]:
final_df.duplicated(subset=['Run Month', 'Key', 'Month Date']).sum()

0

In [963]:
final_df['Calculated Primary Vol'].sum()

3712488.165752257

### Depot PSKU

#### Aggregate to PSKU first

In [964]:
psku_df = final_df.groupby(
    ['PSKU', 'Brand', 'Portfolio', 'Run Month', 'Month Date'],
    as_index=False
)['Calculated Primary Vol'].sum()

In [965]:
psku_df = psku_df[psku_df['Month Date'] >= psku_df['Run Month']]

In [966]:
psku_df

,PSKU,Brand,Portfolio,Run Month,Month Date,Calculated Primary Vol
1,715098,CO_SO_PCP,SKIN CARE,2026-08-31,2026-08-31,0.0
2,715098,CO_SO_PCP,SKIN CARE,2026-08-31,2026-09-30,0.0
3,715098,CO_SO_PCP,SKIN CARE,2026-08-31,2026-10-31,0.0
4,715098,CO_SO_PCP,SKIN CARE,2026-08-31,2026-11-30,0.0
5,715098,CO_SO_PCP,SKIN CARE,2026-08-31,2026-12-31,0.0
...,...,...,...,...,...,...
9795,811564,SAF-MUSLI,FOODS,2026-08-31,2026-12-31,0.0
9796,811564,SAF-MUSLI,FOODS,2026-08-31,2027-01-31,0.0
9797,811564,SAF-MUSLI,FOODS,2026-08-31,2027-02-28,0.0
9798,811564,SAF-MUSLI,FOODS,2026-08-31,2027-03-31,0.0


In [967]:
psku_df['Calculated Primary Vol'].sum()

3712488.1657522568

In [968]:
others_df = others_df.groupby(
    ['parent_material_code', 'material_group_code', 'month_date'], as_index=False
)['Primary P3M'].sum()

len_before_merge = len(others_df)
others_df = others_df.merge(
    brand_md_df.rename(
        columns={'brand_code': 'material_group_code'}
    ),
    on='material_group_code',
    how='left'
)
assert len_before_merge == len(others_df)
del len_before_merge

others_df = others_df.rename(columns={
    'parent_material_code': 'PSKU',
    'material_group_code': 'Brand',
    'month_date': 'Month Date',
    'Primary P3M': 'Calculated Primary Vol',
    'portfolio': 'Portfolio'
})

In [1050]:
others_df['Calculated Primary Vol'].sum()

1358123.369500002

In [970]:
psku_df['Calculated Primary Vol'].min()

0.0

In [971]:
final_others_forecast = pd.DataFrame()

for rm in psku_df['Run Month'].unique():
    tmp = others_df.copy()
    tmp['Run Month'] = rm
    tmp = tmp[tmp['Month Date'] <= psku_df[psku_df['Run Month'] == rm]['Month Date'].max()]
    tmp = tmp[tmp['Month Date'] >= rm]
    final_others_forecast = pd.concat([final_others_forecast, tmp], ignore_index=True)
    del tmp

In [972]:
psku_df = pd.concat(
    [psku_df, final_others_forecast], ignore_index=True
)

In [973]:
psku_df = psku_df.groupby(
    ['PSKU', 'Brand', 'Portfolio', 'Run Month', 'Month Date'],
    as_index=False
)['Calculated Primary Vol'].sum()

In [974]:
psku_df['Calculated Primary Vol'].sum()

3735737.1894189236

In [975]:
depot_psku_primary_query = """
SELECT
    CM.depot_code,
    MM.parent_material_code,
    MM.material_group_code,
    LAST_DAY(MESR.month_date) AS month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain_type = 'E Com B2C' AND 
        chain IN ('Flipkart-National', 'Flipkart-Grocery', 'Big basket B2C', 'RK WORLDINFOCOM', 'Amazon B2C', 
        'FATEHPURIA HYGIENE', 'Nykaa', 'Flipkart-Minutes', 'Purplle', 'Myntra', 'Dealshare', 'FlipkartGrocery', 'City Mall', '1MG', 
        'ARIPL', 'First Cry', 'Meesho', 'RKWorld', 'CITIMALL', 'Firstcry', 'EMAZING DEALS', 'MYNTRA')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
JOIN
(
    SELECT DISTINCT
        customer_code,
        depot_code
    FROM
        mst_customer
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) CM on MESR.distributor_code = CM.customer_code
GROUP BY 1, 2, 3, 4
ORDER BY 1, 3, 2, 4
"""

depot_psku_primary_df = pd.read_sql(
    depot_psku_primary_query,
    prod_conn
)

In [976]:
depot_psku_primary_df

,DEPOT_CODE,PARENT_MATERIAL_CODE,MATERIAL_GROUP_CODE,MONTH_DATE,PRI_ACTUALS_VOL_RUM,PRI_APO_PLAN_VOL_RUM,SEC_APO_PLAN_VOL_RUM,SEC_ACTUALS_VOL_RUM
0,D111,718471,ADV-AHO-R,2017-04-30,0.0,0.000000,NaN,0.0
1,D111,718471,ADV-AHO-R,2017-07-31,0.0,0.000000,NaN,0.0
2,D111,718472,ADV-AHO-R,2017-04-30,0.0,0.000000,NaN,0.0
3,D111,718472,ADV-AHO-R,2017-07-31,0.0,0.000000,NaN,0.0
4,D111,718473,ADV-AHO-R,2017-04-30,0.0,0.000000,NaN,0.0
...,...,...,...,...,...,...,...,...
342387,D677,810179,SW_SGPRF,2026-06-30,0.0,21.831578,0.0,0.0
342388,D677,810179,SW_SGPRF,2026-07-31,0.0,7.823442,0.0,0.0
342389,D677,810179,SW_SGPRF,2026-08-31,0.0,20.021340,0.0,0.0
342390,D677,811169,SW_SGPRF,2026-03-31,0.0,9.818184,0.0,0.0


In [977]:
depot_psku_primary_df.columns = depot_psku_primary_df.columns.str.lower()
depot_psku_primary_df['parent_material_code'] = depot_psku_primary_df['parent_material_code'].astype(int)

In [978]:
depot_psku_primary_df = realign_pskus(depot_psku_primary_df.copy(), 'parent_material_code')

In [979]:
depot_psku_primary_df

,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,D111,718471,ADV-AHO-R,2017-04-30,0.0,0.000000,NaN,0.0
1,D111,718471,ADV-AHO-R,2017-07-31,0.0,0.000000,NaN,0.0
2,D111,718472,ADV-AHO-R,2017-04-30,0.0,0.000000,NaN,0.0
3,D111,718472,ADV-AHO-R,2017-07-31,0.0,0.000000,NaN,0.0
4,D111,718473,ADV-AHO-R,2017-04-30,0.0,0.000000,NaN,0.0
...,...,...,...,...,...,...,...,...
342387,D677,810179,SW_SGPRF,2026-06-30,0.0,21.831578,0.0,0.0
342388,D677,810179,SW_SGPRF,2026-07-31,0.0,7.823442,0.0,0.0
342389,D677,810179,SW_SGPRF,2026-08-31,0.0,20.021340,0.0,0.0
342390,D677,811169,SW_SGPRF,2026-03-31,0.0,9.818184,0.0,0.0


In [980]:
material_master_df = pd.read_sql(
    """select * from mst_material 
    where latest_record_ind=1 and company_code='MIL'""",
    prod_conn
)
material_master_df.columns = material_master_df.columns.str.lower()
assert material_master_df.duplicated(
    subset=['company_code', 'material_code']).sum() == 0

material_master_df['material_code'] = material_master_df['material_code'].astype(np.int64)
material_master_df.duplicated(subset=['material_code', 'parent_material_code', 'material_group_code']).sum()

0

In [981]:
depot_psku_primary_df = depot_psku_primary_df.groupby(
    ['depot_code', 'parent_material_code', 'month_date'],
    as_index=False
)[['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum',
       'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']].sum()

# material_master_df[['parent_material_code', 'brand_code']].dtypes
material_master_df['parent_material_code'] = material_master_df['parent_material_code'].astype(int)
material_master_df.loc[material_master_df['parent_material_code'].isin([725930,731857]), 'material_group_code'] = 'H&C_ALMND'

# offtake_df.drop(columns = ['brand_code'],inplace = True)
len_before_merge = len(depot_psku_primary_df)
depot_psku_primary_df = depot_psku_primary_df.merge(
    material_master_df[['parent_material_code', 'material_group_code']].drop_duplicates(),
    left_on=['parent_material_code'],right_on = ['parent_material_code'],
    how='left'
)
assert len_before_merge == len(depot_psku_primary_df)

In [982]:
depot_psku_primary_df = depot_psku_primary_df.groupby(
    ['depot_code', 'parent_material_code', 'material_group_code', 'month_date'], as_index=False, dropna=False
).sum()

In [983]:
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['parent_material_code'] != 715096]

In [984]:
depot_psku_primary_df.duplicated(subset=['depot_code', 'parent_material_code', 'month_date']).sum()

0

In [985]:
depot_psku_primary_df.shape

(335113, 8)

In [986]:
# # For duplicates, keep only the row with PABABY_ML material_group_code
# duplicates = depot_psku_primary_df[depot_psku_primary_df.duplicated(subset=['depot_code', 'parent_material_code', 'month_date'], keep=False)]

# # Get indices of duplicates that are NOT PABABY_ML
# indices_to_drop = duplicates[duplicates['material_group_code'] != 'PABABY_ML'].index

# # Remove those rows
# depot_psku_primary_df = depot_psku_primary_df.drop(indices_to_drop)

# # Verify no duplicates remain
# print(depot_psku_primary_df.duplicated(subset=['depot_code', 'parent_material_code', 'month_date']).sum())

In [987]:
depot_psku_primary_df.shape

(335113, 8)

In [988]:
depot_psku_primary_df = impute_missing_dates(
    depot_psku_primary_df.copy(),
    key=['depot_code', 'parent_material_code'],
    date_col='month_date'
)

19004it [00:04, 4035.61it/s]


In [989]:
cols = ['depot_code', 'parent_material_code', 'material_group_code']

depot_psku_primary_df[cols] = depot_psku_primary_df.groupby('key')[cols].transform(lambda x: x.ffill().bfill())

In [990]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    depot_psku_primary_df[col] = depot_psku_primary_df[col].fillna(0)

In [991]:
depot_psku_primary_df['parent_material_code'] = depot_psku_primary_df['parent_material_code'].astype(int)

In [992]:
depot_psku_primary_df.duplicated(subset=['key', 'month_date']).sum()


0

In [993]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if depot_psku_primary_df[col].min() < 0:
        print(col)

for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    depot_psku_primary_df[col] = depot_psku_primary_df[col].clip(lower=0)



pri_actuals_vol_rum
sec_actuals_vol_rum


In [994]:
depot_psku_primary_df.sort_values(by=['key', 'month_date'], inplace=True)

In [995]:
depot_psku_primary_df['Primary P3M Vol'] = depot_psku_primary_df.groupby(
    ['key'], 
    as_index = False, group_keys = False
)['pri_actuals_vol_rum'].shift(1).rolling(window=3, min_periods=1).mean()

In [996]:
depot_psku_primary_df['Primary Actuals Lag 1 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(1)

depot_psku_primary_df['Primary Actuals Lag 2 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(2)

depot_psku_primary_df['Primary Actuals Lag 3 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(3)

depot_psku_primary_df['LY Primary Actuals Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(12)

depot_psku_primary_df['LY Primary Actuals Lag 1 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(13)

depot_psku_primary_df['LY Primary Actuals Lag 2 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(14)

depot_psku_primary_df['LY Primary Actuals Lag 3 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(15)


depot_psku_primary_df['LY Primary Actuals Lead 1 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(11)

depot_psku_primary_df['LY Primary Actuals Lead 2 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(10)

In [997]:
depot_psku_primary_df['LY Primary P3M Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['Primary P3M Vol'].shift(12)

In [998]:
depot_psku_primary_df = depot_psku_primary_df.rename(columns={
    'key': 'Key',
    'depot_code': 'Depot',
    'parent_material_code': 'PSKU',
    'material_group_code': 'Brand',
    'month_date': 'Month Date',
    'pri_actuals_vol_rum': 'Primary Actuals Vol',
    'pri_apo_plan_vol_rum': 'Primary Plan Vol',
    'sec_actuals_vol_rum': 'Secondary Actuals Vol',
    'sec_apo_plan_vol_rum': 'Secondary Plan Vol'
})

In [999]:
depot_psku_primary_df

,Month Date,Key,Depot,PSKU,Brand,Primary Actuals Vol,Primary Plan Vol,Secondary Plan Vol,Secondary Actuals Vol,Primary P3M Vol,Primary Actuals Lag 1 Vol,Primary Actuals Lag 2 Vol,Primary Actuals Lag 3 Vol,LY Primary Actuals Vol,LY Primary Actuals Lag 1 Vol,LY Primary Actuals Lag 2 Vol,LY Primary Actuals Lag 3 Vol,LY Primary Actuals Lead 1 Vol,LY Primary Actuals Lead 2 Vol,LY Primary P3M Vol
0,2017-04-30,D111_709567,D111,709567,SAFF OATS,0.0,0.0000,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2017-05-31,D111_709567,D111,709567,SAFF OATS,0.0,0.0000,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2017-06-30,D111_709567,D111,709567,SAFF OATS,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2017-07-31,D111_709567,D111,709567,SAFF OATS,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2017-08-31,D111_709567,D111,709567,SAFF OATS,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
961897,2026-08-31,D677_811564,D677,811564,SAF-MUSLI,0.0,0.0808,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
961898,2026-09-30,D677_811564,D677,811564,SAF-MUSLI,0.0,0.0000,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
961899,2026-10-31,D677_811564,D677,811564,SAF-MUSLI,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
961900,2026-11-30,D677_811564,D677,811564,SAF-MUSLI,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [1000]:
final_depot_psku_df = depot_psku_primary_df[['Key', 'Depot', 'PSKU', 'Brand']].drop_duplicates()

In [1001]:
psku_df['Calculated Primary Vol'].sum()

3735737.1894189236

In [1002]:
x = psku_df.copy()

In [1003]:
# # For duplicates, keep only the row with PABABY_ML material_group_code
# duplicates = psku_df[psku_df.duplicated(subset=['PSKU','Run Month', 'Month Date'], keep=False)]

# # Get indices of duplicates that are NOT PABABY_ML
# indices_to_drop = duplicates[duplicates['Brand'] != 'PABABY_ML'].index

# # Remove those rows
# psku_df = psku_df.drop(indices_to_drop)

# # Verify no duplicates remain
# print(psku_df.duplicated(subset=['PSKU','Run Month', 'Month Date']).sum())

In [1004]:
psku_df['Calculated Primary Vol'].sum()

3735737.1894189236

In [1005]:
tmp_df = pd.DataFrame()

for rm in ['2026-08-31']: 
    mth_dates = [pd.to_datetime(rm) + MonthEnd(i) for i in range(-1, 9)]
    for mth_dt in mth_dates:
        tmp_df2 = final_depot_psku_df.copy()
        tmp_df2['Run Month'] = pd.to_datetime(rm)
        tmp_df2['Month Date'] = pd.to_datetime(mth_dt)

        tmp_df = pd.concat([tmp_df, tmp_df2], ignore_index=True)
        del tmp_df2

final_depot_psku_df = tmp_df.copy()    
del tmp_df

In [1006]:
psku_df

,PSKU,Brand,Portfolio,Run Month,Month Date,Calculated Primary Vol
0,715098,CO_SO_PCP,SKIN CARE,2026-08-31,2026-08-31,0.0
1,715098,CO_SO_PCP,SKIN CARE,2026-08-31,2026-09-30,0.0
2,715098,CO_SO_PCP,SKIN CARE,2026-08-31,2026-10-31,0.0
3,715098,CO_SO_PCP,SKIN CARE,2026-08-31,2026-11-30,0.0
4,715098,CO_SO_PCP,SKIN CARE,2026-08-31,2026-12-31,0.0
...,...,...,...,...,...,...
8840,811564,SAF-MUSLI,FOODS,2026-08-31,2026-12-31,0.0
8841,811564,SAF-MUSLI,FOODS,2026-08-31,2027-01-31,0.0
8842,811564,SAF-MUSLI,FOODS,2026-08-31,2027-02-28,0.0
8843,811564,SAF-MUSLI,FOODS,2026-08-31,2027-03-31,0.0


In [1007]:
len_before_merge = len(final_depot_psku_df)
final_depot_psku_df = final_depot_psku_df.merge(
    psku_df[['PSKU', 'Run Month', 'Month Date', 'Calculated Primary Vol']], 
    on=['PSKU', 'Run Month', 'Month Date'],
    how='left'
)
assert len_before_merge == len(final_depot_psku_df)
del len_before_merge

In [1008]:
final_depot_psku_df['Calculated Primary Vol'].sum()

86256847.24094369

In [1009]:
final_depot_psku_df

,Key,Depot,PSKU,Brand,Run Month,Month Date,Calculated Primary Vol
0,D111_709567,D111,709567,SAFF OATS,2026-08-31,2026-07-31,NaN
1,D111_710981,D111,710981,PA_MEN_AH,2026-08-31,2026-07-31,NaN
2,D111_711040,D111,711040,PCNO GOLD,2026-08-31,2026-07-31,NaN
3,D111_711041,D111,711041,PCNO GOLD,2026-08-31,2026-07-31,NaN
4,D111_711042,D111,711042,PCNO GOLD,2026-08-31,2026-07-31,NaN
...,...,...,...,...,...,...,...
190035,D677_811269,D677,811269,PA_ESS_HO,2026-08-31,2027-04-30,0.000000
190036,D677_811279,D677,811279,SAF_CDPRS,2026-08-31,2027-04-30,0.000000
190037,D677_811287,D677,811287,PA_RSW_SR,2026-08-31,2027-04-30,96.157549
190038,D677_811416,D677,811416,SAF-MUSLI,2026-08-31,2027-04-30,0.000000


In [1010]:
# x = final_depot_psku_df.groupby(['PSKU','Month Date'])['Calculated Primary Vol'].mean().reset_index()
# x[x['Month Date']>'2026-06-30']['Calculated Primary Vol'].sum()

In [1011]:
depot_psku_primary_df.columns

Index(['Month Date', 'Key', 'Depot', 'PSKU', 'Brand', 'Primary Actuals Vol',
       'Primary Plan Vol', 'Secondary Plan Vol', 'Secondary Actuals Vol',
       'Primary P3M Vol', 'Primary Actuals Lag 1 Vol',
       'Primary Actuals Lag 2 Vol', 'Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Vol', 'LY Primary Actuals Lag 1 Vol',
       'LY Primary Actuals Lag 2 Vol', 'LY Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Lead 1 Vol', 'LY Primary Actuals Lead 2 Vol',
       'LY Primary P3M Vol'],
      dtype='object')

In [1012]:
len_before_merge = len(final_depot_psku_df)
final_depot_psku_df = final_depot_psku_df.merge(
    depot_psku_primary_df[['Depot', 'PSKU', 'Month Date', 'Primary Actuals Vol',
       'Primary Plan Vol', 'Secondary Plan Vol', 'Secondary Actuals Vol',
       'Primary P3M Vol', 'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol',
       'Primary Actuals Lag 3 Vol', 'LY Primary Actuals Vol',  'LY Primary Actuals Lag 1 Vol',
       'LY Primary Actuals Lag 2 Vol', 'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol', 
       'LY Primary Actuals Lead 2 Vol',
       'LY Primary P3M Vol']], 
    on=['Depot', 'PSKU', 'Month Date'],
    how='left'
)
assert len_before_merge == len(final_depot_psku_df)
del len_before_merge

In [1013]:
final_depot_psku_df

,Key,Depot,PSKU,Brand,Run Month,Month Date,Calculated Primary Vol,Primary Actuals Vol,Primary Plan Vol,Secondary Plan Vol,...,Primary Actuals Lag 1 Vol,Primary Actuals Lag 2 Vol,Primary Actuals Lag 3 Vol,LY Primary Actuals Vol,LY Primary Actuals Lag 1 Vol,LY Primary Actuals Lag 2 Vol,LY Primary Actuals Lag 3 Vol,LY Primary Actuals Lead 1 Vol,LY Primary Actuals Lead 2 Vol,LY Primary P3M Vol
0,D111_709567,D111,709567,SAFF OATS,2026-08-31,2026-07-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,D111_710981,D111,710981,PA_MEN_AH,2026-08-31,2026-07-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,D111_711040,D111,711040,PCNO GOLD,2026-08-31,2026-07-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,D111_711041,D111,711041,PCNO GOLD,2026-08-31,2026-07-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,D111_711042,D111,711042,PCNO GOLD,2026-08-31,2026-07-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190035,D677_811269,D677,811269,PA_ESS_HO,2026-08-31,2027-04-30,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
190036,D677_811279,D677,811279,SAF_CDPRS,2026-08-31,2027-04-30,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
190037,D677_811287,D677,811287,PA_RSW_SR,2026-08-31,2027-04-30,96.157549,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
190038,D677_811416,D677,811416,SAF-MUSLI,2026-08-31,2027-04-30,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [1014]:
final_depot_psku_df.columns

Index(['Key', 'Depot', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Calculated Primary Vol', 'Primary Actuals Vol', 'Primary Plan Vol',
       'Secondary Plan Vol', 'Secondary Actuals Vol', 'Primary P3M Vol',
       'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol',
       'Primary Actuals Lag 3 Vol', 'LY Primary Actuals Vol',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
       'LY Primary Actuals Lead 2 Vol', 'LY Primary P3M Vol'],
      dtype='object')

In [1015]:
final_depot_psku_df['Primary P3M copy Vol'] = final_depot_psku_df['Primary P3M Vol'].copy()

In [1016]:
final_depot_psku_df['Primary P3M Vol'] = final_depot_psku_df['Primary P3M Vol'].fillna(0)

In [1017]:
final_depot_psku_df

,Key,Depot,PSKU,Brand,Run Month,Month Date,Calculated Primary Vol,Primary Actuals Vol,Primary Plan Vol,Secondary Plan Vol,...,Primary Actuals Lag 2 Vol,Primary Actuals Lag 3 Vol,LY Primary Actuals Vol,LY Primary Actuals Lag 1 Vol,LY Primary Actuals Lag 2 Vol,LY Primary Actuals Lag 3 Vol,LY Primary Actuals Lead 1 Vol,LY Primary Actuals Lead 2 Vol,LY Primary P3M Vol,Primary P3M copy Vol
0,D111_709567,D111,709567,SAFF OATS,2026-08-31,2026-07-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,D111_710981,D111,710981,PA_MEN_AH,2026-08-31,2026-07-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,D111_711040,D111,711040,PCNO GOLD,2026-08-31,2026-07-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,D111_711041,D111,711041,PCNO GOLD,2026-08-31,2026-07-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,D111_711042,D111,711042,PCNO GOLD,2026-08-31,2026-07-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190035,D677_811269,D677,811269,PA_ESS_HO,2026-08-31,2027-04-30,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
190036,D677_811279,D677,811279,SAF_CDPRS,2026-08-31,2027-04-30,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
190037,D677_811287,D677,811287,PA_RSW_SR,2026-08-31,2027-04-30,96.157549,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
190038,D677_811416,D677,811416,SAF-MUSLI,2026-08-31,2027-04-30,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [1018]:
final_depot_psku_df['PSKU Primary P3M Sum Vol'] = final_depot_psku_df.groupby(
    ['Run Month', 'Month Date', 'PSKU']
)['Primary P3M Vol'].transform('sum')

In [1019]:
final_depot_psku_df

,Key,Depot,PSKU,Brand,Run Month,Month Date,Calculated Primary Vol,Primary Actuals Vol,Primary Plan Vol,Secondary Plan Vol,...,Primary Actuals Lag 3 Vol,LY Primary Actuals Vol,LY Primary Actuals Lag 1 Vol,LY Primary Actuals Lag 2 Vol,LY Primary Actuals Lag 3 Vol,LY Primary Actuals Lead 1 Vol,LY Primary Actuals Lead 2 Vol,LY Primary P3M Vol,Primary P3M copy Vol,PSKU Primary P3M Sum Vol
0,D111_709567,D111,709567,SAFF OATS,2026-08-31,2026-07-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,D111_710981,D111,710981,PA_MEN_AH,2026-08-31,2026-07-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,D111_711040,D111,711040,PCNO GOLD,2026-08-31,2026-07-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,D111_711041,D111,711041,PCNO GOLD,2026-08-31,2026-07-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,D111_711042,D111,711042,PCNO GOLD,2026-08-31,2026-07-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190035,D677_811269,D677,811269,PA_ESS_HO,2026-08-31,2027-04-30,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
190036,D677_811279,D677,811279,SAF_CDPRS,2026-08-31,2027-04-30,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
190037,D677_811287,D677,811287,PA_RSW_SR,2026-08-31,2027-04-30,96.157549,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
190038,D677_811416,D677,811416,SAF-MUSLI,2026-08-31,2027-04-30,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [1020]:
final_depot_psku_df.sort_values(by=['Run Month', 'Key', 'Month Date'], inplace=True)

In [1021]:
final_depot_psku_df.columns

Index(['Key', 'Depot', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Calculated Primary Vol', 'Primary Actuals Vol', 'Primary Plan Vol',
       'Secondary Plan Vol', 'Secondary Actuals Vol', 'Primary P3M Vol',
       'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol',
       'Primary Actuals Lag 3 Vol', 'LY Primary Actuals Vol',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
       'LY Primary Actuals Lead 2 Vol', 'LY Primary P3M Vol',
       'Primary P3M copy Vol', 'PSKU Primary P3M Sum Vol'],
      dtype='object')

In [1022]:
for col in ['Primary P3M Vol', 'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol', 
            'Primary Actuals Lag 3 Vol', 'PSKU Primary P3M Sum Vol']:
    # if not 'LY' in col:  'LY P6M',
    final_depot_psku_df.loc[final_depot_psku_df['Month Date'] > final_depot_psku_df['Run Month'], [col]] = np.nan
    final_depot_psku_df[col] = final_depot_psku_df.groupby(['Run Month', 'Key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [1023]:
final_depot_psku_df

,Key,Depot,PSKU,Brand,Run Month,Month Date,Calculated Primary Vol,Primary Actuals Vol,Primary Plan Vol,Secondary Plan Vol,...,Primary Actuals Lag 3 Vol,LY Primary Actuals Vol,LY Primary Actuals Lag 1 Vol,LY Primary Actuals Lag 2 Vol,LY Primary Actuals Lag 3 Vol,LY Primary Actuals Lead 1 Vol,LY Primary Actuals Lead 2 Vol,LY Primary P3M Vol,Primary P3M copy Vol,PSKU Primary P3M Sum Vol
0,D111_709567,D111,709567,SAFF OATS,2026-08-31,2026-07-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19004,D111_709567,D111,709567,SAFF OATS,2026-08-31,2026-08-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
38008,D111_709567,D111,709567,SAFF OATS,2026-08-31,2026-09-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
57012,D111_709567,D111,709567,SAFF OATS,2026-08-31,2026-10-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
76016,D111_709567,D111,709567,SAFF OATS,2026-08-31,2026-11-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114023,D677_811564,D677,811564,SAF-MUSLI,2026-08-31,2026-12-31,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
133027,D677_811564,D677,811564,SAF-MUSLI,2026-08-31,2027-01-31,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
152031,D677_811564,D677,811564,SAF-MUSLI,2026-08-31,2027-02-28,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
171035,D677_811564,D677,811564,SAF-MUSLI,2026-08-31,2027-03-31,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [1024]:
final_depot_psku_df.columns

Index(['Key', 'Depot', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Calculated Primary Vol', 'Primary Actuals Vol', 'Primary Plan Vol',
       'Secondary Plan Vol', 'Secondary Actuals Vol', 'Primary P3M Vol',
       'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol',
       'Primary Actuals Lag 3 Vol', 'LY Primary Actuals Vol',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
       'LY Primary Actuals Lead 2 Vol', 'LY Primary P3M Vol',
       'Primary P3M copy Vol', 'PSKU Primary P3M Sum Vol'],
      dtype='object')

In [1025]:
final_depot_psku_df

,Key,Depot,PSKU,Brand,Run Month,Month Date,Calculated Primary Vol,Primary Actuals Vol,Primary Plan Vol,Secondary Plan Vol,...,Primary Actuals Lag 3 Vol,LY Primary Actuals Vol,LY Primary Actuals Lag 1 Vol,LY Primary Actuals Lag 2 Vol,LY Primary Actuals Lag 3 Vol,LY Primary Actuals Lead 1 Vol,LY Primary Actuals Lead 2 Vol,LY Primary P3M Vol,Primary P3M copy Vol,PSKU Primary P3M Sum Vol
0,D111_709567,D111,709567,SAFF OATS,2026-08-31,2026-07-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19004,D111_709567,D111,709567,SAFF OATS,2026-08-31,2026-08-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
38008,D111_709567,D111,709567,SAFF OATS,2026-08-31,2026-09-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
57012,D111_709567,D111,709567,SAFF OATS,2026-08-31,2026-10-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
76016,D111_709567,D111,709567,SAFF OATS,2026-08-31,2026-11-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114023,D677_811564,D677,811564,SAF-MUSLI,2026-08-31,2026-12-31,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
133027,D677_811564,D677,811564,SAF-MUSLI,2026-08-31,2027-01-31,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
152031,D677_811564,D677,811564,SAF-MUSLI,2026-08-31,2027-02-28,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
171035,D677_811564,D677,811564,SAF-MUSLI,2026-08-31,2027-03-31,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [1026]:
final_depot_psku_df['PSKU P3M Contribution'] = final_depot_psku_df['Primary P3M Vol'] / final_depot_psku_df['PSKU Primary P3M Sum Vol']

In [1027]:
final_depot_psku_df['PSKU P3M Contribution'].isnull().sum()

111538

In [1028]:
# x = final_depot_psku_df.groupby(['PSKU','Month Date'])['Calculated Primary Vol'].mean().reset_index()
# x[x['Month Date']>'2026-06-30']['Calculated Primary Vol'].sum()

In [1029]:
final_depot_psku_df['Calculated Depot PSKU Primary Vol'] = final_depot_psku_df['Calculated Primary Vol'] \
    * final_depot_psku_df['PSKU P3M Contribution']

In [1030]:
final_depot_psku_df.rename(columns={'Calculated Primary Vol': 'Calculated PSKU Primary Vol'}, inplace=True)

In [1031]:
final_depot_psku_df['Calculated Depot PSKU Primary Vol'].sum()

3711755.0559739764

In [1032]:
vol_to_val_cols = [col for col in final_depot_psku_df.columns if ('Vol' in col) and ('copy' not in col)]
vol_to_val_cols

['Calculated PSKU Primary Vol',
 'Primary Actuals Vol',
 'Primary Plan Vol',
 'Secondary Plan Vol',
 'Secondary Actuals Vol',
 'Primary P3M Vol',
 'Primary Actuals Lag 1 Vol',
 'Primary Actuals Lag 2 Vol',
 'Primary Actuals Lag 3 Vol',
 'LY Primary Actuals Vol',
 'LY Primary Actuals Lag 1 Vol',
 'LY Primary Actuals Lag 2 Vol',
 'LY Primary Actuals Lag 3 Vol',
 'LY Primary Actuals Lead 1 Vol',
 'LY Primary Actuals Lead 2 Vol',
 'LY Primary P3M Vol',
 'PSKU Primary P3M Sum Vol',
 'Calculated Depot PSKU Primary Vol']

In [1033]:
qtr_ind_rate_df.head()

,month_date,brand_code,qtr_ind_rate
0,2027-03-31,CMX_WELPD,3014.000
1,2027-03-31,4700_BCPC,222.000
2,2027-03-31,CMX_PRTPD,3014.000
3,2027-03-31,PA_CN_HGO,488.152
4,2027-03-31,TRU_RAWDF,800.000


In [1034]:
len_before_merge = len(final_depot_psku_df)
final_depot_psku_df = final_depot_psku_df.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(columns={
        'brand_code': 'Brand',
        'qtr_ind_rate': 'Index Rate'
    }),
    on=['Brand'],
    how='left'
)
assert len_before_merge == len(final_depot_psku_df)
del len_before_merge

In [1035]:
for col in vol_to_val_cols:
    final_depot_psku_df[col[:-3] + 'Val'] = final_depot_psku_df[col] * final_depot_psku_df['Index Rate'] / (10 ** 7)

In [1036]:
final_depot_psku_df

,Key,Depot,PSKU,Brand,Run Month,Month Date,Calculated PSKU Primary Vol,Primary Actuals Vol,Primary Plan Vol,Secondary Plan Vol,...,Primary Actuals Lag 3 Val,LY Primary Actuals Val,LY Primary Actuals Lag 1 Val,LY Primary Actuals Lag 2 Val,LY Primary Actuals Lag 3 Val,LY Primary Actuals Lead 1 Val,LY Primary Actuals Lead 2 Val,LY Primary P3M Val,PSKU Primary P3M Sum Val,Calculated Depot PSKU Primary Val
0,D111_709567,D111,709567,SAFF OATS,2026-08-31,2026-07-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
1,D111_709567,D111,709567,SAFF OATS,2026-08-31,2026-08-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
2,D111_709567,D111,709567,SAFF OATS,2026-08-31,2026-09-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
3,D111_709567,D111,709567,SAFF OATS,2026-08-31,2026-10-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
4,D111_709567,D111,709567,SAFF OATS,2026-08-31,2026-11-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190035,D677_811564,D677,811564,SAF-MUSLI,2026-08-31,2026-12-31,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
190036,D677_811564,D677,811564,SAF-MUSLI,2026-08-31,2027-01-31,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
190037,D677_811564,D677,811564,SAF-MUSLI,2026-08-31,2027-02-28,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
190038,D677_811564,D677,811564,SAF-MUSLI,2026-08-31,2027-03-31,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN


In [1037]:
final_depot_psku_df['Calculated Depot PSKU Primary Val'] = final_depot_psku_df['Calculated Depot PSKU Primary Val'].fillna(0)

In [1038]:
final_depot_psku_df['M Month'] = final_depot_psku_df.apply(
    lambda x: mappings[x['Run Month']].get(x['Month Date'], np.nan),
    axis=1
)

In [1039]:
final_depot_psku_df = final_depot_psku_df[final_depot_psku_df['M Month'].notna()]

In [1040]:
final_depot_psku_df.groupby(
    ['Run Month', 'Month Date', 'PSKU']
)['PSKU P3M Contribution'].max().max()

1.0

In [1041]:
final_df.columns

Index(['Key', 'Chain', 'PSKU', 'Brand', 'Index Rate', 'Portfolio', 'Run Month',
       'Month Date', 'M month', 'Primary Till Date Actuals Vol',
       'Secondary Plan Vol', 'Primary P3M Vol', 'Offtake Chain PSKU Vol',
       'Offtake Chain PSKU Forecast Vol', 'Norms SOH', 'Norm Days',
       'Safety Stock Vol', 'Actual Closing SOH Vol',
       'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol',
       'Assumed Closing SOH Vol', 'Assumed Closing SOH Lag 1 Vol',
       'Final Assumed Closing SOH Vol', 'Final Assumed Closing SOH Lag 1 Vol',
       'Primary Actuals Vol', 'Sec Actuals Vol', 'Primary P3M redundant Vol',
       'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M Vol',
       'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol',
       'Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lag 1 Vol',
       'LY Primary Actuals Lag 2 Vol', 'LY Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Lead 1 Vol', 'LY Primary Actuals Lead 2 Vol',
       'Off

In [1042]:
final_df['Calculated Primary Val'].sum(), final_depot_psku_df['Calculated Depot PSKU Primary Val'].sum()

(411.64627575143714, 410.54188559605967)

In [1046]:
final_df[final_df['Month Date']=='2026-09-30']['Calculated Primary Val'].sum()

51.18818563457459

In [1047]:
final_df['Chain'].unique()

array(['Amazon ARIPL', 'Amazon RK', 'Big Basket', 'Flipkart Grocery',
       'Flipkart National', 'Meesho', 'Myntra', 'Nykaa'], dtype=object)

In [1049]:
final_depot_psku_df.isnull().sum()

Key                                       0
Depot                                     0
PSKU                                      0
Brand                                     0
Run Month                                 0
Month Date                                0
Calculated PSKU Primary Vol           28383
Primary Actuals Vol                   76016
Primary Plan Vol                      76016
Secondary Plan Vol                    76016
Secondary Actuals Vol                 76016
Primary P3M Vol                           0
Primary Actuals Lag 1 Vol               423
Primary Actuals Lag 2 Vol              1449
Primary Actuals Lag 3 Vol              2331
LY Primary Actuals Vol                87011
LY Primary Actuals Lag 1 Vol          88341
LY Primary Actuals Lag 2 Vol          89505
LY Primary Actuals Lag 3 Vol          90637
LY Primary Actuals Lead 1 Vol         85819
LY Primary Actuals Lead 2 Vol         84690
LY Primary P3M Vol                    87011
Primary P3M copy Vol            

In [1092]:
final_df[final_df['Month Date']=='2026-09-30']['Offtake Actuals Lag 1 Val'].sum()

42.49894498911772

In [1057]:
final_df

,Key,Chain,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,M month,Primary Till Date Actuals Vol,...,LY Offtake P3M Val,Offtake Actuals Lag 1 Val,Offtake Actuals Lag 2 Val,Offtake Actuals Lag 3 Val,LY Offtake Actuals Lag 1 Val,LY Offtake Actuals Lag 2 Val,LY Offtake Actuals Lag 3 Val,LY Offtake Actuals Lead 1 Val,LY Offtake Actuals Lead 2 Val,Calculated Primary Val
0,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1220.081000,SKIN CARE,2026-08-31,2026-07-31,NaN,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1220.081000,SKIN CARE,2026-08-31,2026-08-31,M,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1220.081000,SKIN CARE,2026-08-31,2026-09-30,M+1,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1220.081000,SKIN CARE,2026-08-31,2026-10-31,M+2,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1220.081000,SKIN CARE,2026-08-31,2026-11-30,M+3,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60915,Nykaa_811564,Nykaa,811564,SAF-MUSLI,315513.490535,FOODS,2026-08-31,2026-12-31,M+4,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
60916,Nykaa_811564,Nykaa,811564,SAF-MUSLI,315513.490535,FOODS,2026-08-31,2027-01-31,M+5,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
60917,Nykaa_811564,Nykaa,811564,SAF-MUSLI,315513.490535,FOODS,2026-08-31,2027-02-28,M+6,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
60918,Nykaa_811564,Nykaa,811564,SAF-MUSLI,315513.490535,FOODS,2026-08-31,2027-03-31,M+7,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [1059]:
remain = lags_final_offtakes_historical_df[['key', 'month_date', 'offtake_vol_rum', 'OT_Lag_2', 'OT_Lag_3']].rename(
    columns = {
        'month_date': 'run_month',
        'offtake_vol_rum': 'Offtake Actuals Lag 1 Vol',
        'OT_Lag_2': 'Offtake Actuals Lag 2 Vol',
        'OT_Lag_3': 'Offtake Actuals Lag 3 Vol'}
)
remain = remain[remain['run_month'] == '2026-08-31']
remain = remain[~remain['key'].isin(final_df['Key'].unique())]

# Extract parent_material_code from the last 6 digits of the 'key' column
remain['parent_material_code'] = remain['key'].str[-6:]

remain

,key,run_month,Offtake Actuals Lag 1 Vol,Offtake Actuals Lag 2 Vol,Offtake Actuals Lag 3 Vol,parent_material_code
957,Amazon RK_710542,2026-08-31,0.0000,0.0000,0.0000,710542
1096,Amazon RK_718301,2026-08-31,0.0000,0.0000,0.0000,718301
2366,Amazon RK_718458,2026-08-31,0.0000,0.0000,0.0000,718458
2609,Amazon RK_718466,2026-08-31,0.0000,0.0000,0.0000,718466
2815,Amazon RK_718477,2026-08-31,0.0000,0.0000,0.0000,718477
...,...,...,...,...,...,...
138829,Purplle_807725,2026-08-31,0.0000,0.0000,0.0000,807725
138852,Purplle_809042,2026-08-31,0.0000,0.0000,0.0000,809042
138887,Purplle_809250,2026-08-31,0.0005,0.0014,0.0009,809250
138903,Purplle_810673,2026-08-31,0.1680,0.0560,0.0980,810673


In [1064]:
material_master_df.rename(columns={'material_group_code': 'brand_code'}, inplace=True)
remain['parent_material_code'] = remain['parent_material_code'].astype(int)

In [1065]:
len_before_merge = len(remain)
remain = remain.merge(
    material_master_df[['parent_material_code', 'brand_code']].drop_duplicates(),
    left_on=['parent_material_code'],right_on = ['parent_material_code'],
    how='left'
)
assert len_before_merge == len(remain)

def read_qtr_ind_rate_table():
    """
    Fetch the club sku information from  DWH_SAP_INDEX_TURNOVER_MONTHWISE table.

    Return:
        qtr_ind_rate_data: pandas dataframe
        - dataframe contains all the results from the index rate table.
    """
    connection = get_dbconnection(db_name='PROD')
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=connection, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    connection.close()
    return qtr_ind_rate


qtr_df = read_qtr_ind_rate_table()
qtr_df.columns = qtr_df.columns.str.lower()
qtr_df.head()


len_before_merge = len(remain)

remain = remain.rename(columns={'material_group_code': 'brand_code'}).merge(
    qtr_df.drop('month_date', axis=1),
    on=['brand_code'],
    how='left'
)

assert len_before_merge == len(remain)



Credentials retrieved successfully for prod db.


In [1067]:
remain['Offtake Actuals Lag 1 Val'] = remain['Offtake Actuals Lag 1 Vol'] * remain['qtr_ind_rate'] / (10 ** 7)
remain['Offtake Actuals Lag 2 Val'] = remain['Offtake Actuals Lag 2 Vol'] * remain['qtr_ind_rate'] / (10 ** 7)
remain['Offtake Actuals Lag 3 Val'] = remain['Offtake Actuals Lag 3 Vol'] * remain['qtr_ind_rate'] / (10 ** 7)
remain

,key,run_month,Offtake Actuals Lag 1 Vol,Offtake Actuals Lag 2 Vol,Offtake Actuals Lag 3 Vol,parent_material_code,brand_code,qtr_ind_rate,Offtake Actuals Lag 1 Val,Offtake Actuals Lag 2 Val,Offtake Actuals Lag 3 Val
0,Amazon RK_710542,2026-08-31,0.0000,0.0000,0.0000,710542,PCNO(R),3.492740e+05,0.000000,0.000000,0.000000
1,Amazon RK_718301,2026-08-31,0.0000,0.0000,0.0000,718301,PCNO FLEX,2.300000e+05,0.000000,0.000000,0.000000
2,Amazon RK_718458,2026-08-31,0.0000,0.0000,0.0000,718458,NHR-SABDM,1.884937e+02,0.000000,0.000000,0.000000
3,Amazon RK_718466,2026-08-31,0.0000,0.0000,0.0000,718466,PCNO(R),3.492740e+05,0.000000,0.000000,0.000000
4,Amazon RK_718477,2026-08-31,0.0000,0.0000,0.0000,718477,NHR-UTTAM,2.500000e+05,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
524,Purplle_807725,2026-08-31,0.0000,0.0000,0.0000,807725,BIO OILS,3.790534e+03,0.000000,0.000000,0.000000
525,Purplle_809042,2026-08-31,0.0000,0.0000,0.0000,809042,PABABY_GM,3.664850e+02,0.000000,0.000000,0.000000
526,Purplle_809250,2026-08-31,0.0005,0.0014,0.0009,809250,LVNPST_ML,2.892133e+06,0.000145,0.000405,0.000260
527,Purplle_810673,2026-08-31,0.1680,0.0560,0.0980,810673,PA_ESS_HO,1.286063e+04,0.000216,0.000072,0.000126


In [1075]:
remain = remain.rename(columns={
    'key': 'Key',
    'run_month': 'Run Month',
    'parent_material_code': 'PSKU',
    'brand_code': 'Brand',
    'qtr_ind_rate': 'Index Rate'
})

remain

,Key,Run Month,Offtake Actuals Lag 1 Vol,Offtake Actuals Lag 2 Vol,Offtake Actuals Lag 3 Vol,PSKU,Brand,Index Rate,Offtake Actuals Lag 1 Val,Offtake Actuals Lag 2 Val,Offtake Actuals Lag 3 Val
0,Amazon RK_710542,2026-08-31,0.0000,0.0000,0.0000,710542,PCNO(R),3.492740e+05,0.000000,0.000000,0.000000
1,Amazon RK_718301,2026-08-31,0.0000,0.0000,0.0000,718301,PCNO FLEX,2.300000e+05,0.000000,0.000000,0.000000
2,Amazon RK_718458,2026-08-31,0.0000,0.0000,0.0000,718458,NHR-SABDM,1.884937e+02,0.000000,0.000000,0.000000
3,Amazon RK_718466,2026-08-31,0.0000,0.0000,0.0000,718466,PCNO(R),3.492740e+05,0.000000,0.000000,0.000000
4,Amazon RK_718477,2026-08-31,0.0000,0.0000,0.0000,718477,NHR-UTTAM,2.500000e+05,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
524,Purplle_807725,2026-08-31,0.0000,0.0000,0.0000,807725,BIO OILS,3.790534e+03,0.000000,0.000000,0.000000
525,Purplle_809042,2026-08-31,0.0000,0.0000,0.0000,809042,PABABY_GM,3.664850e+02,0.000000,0.000000,0.000000
526,Purplle_809250,2026-08-31,0.0005,0.0014,0.0009,809250,LVNPST_ML,2.892133e+06,0.000145,0.000405,0.000260
527,Purplle_810673,2026-08-31,0.1680,0.0560,0.0980,810673,PA_ESS_HO,1.286063e+04,0.000216,0.000072,0.000126


In [1077]:
final_df.columns

Index(['Key', 'Chain', 'PSKU', 'Brand', 'Index Rate', 'Portfolio', 'Run Month',
       'Month Date', 'M month', 'Primary Till Date Actuals Vol',
       'Secondary Plan Vol', 'Primary P3M Vol', 'Offtake Chain PSKU Vol',
       'Offtake Chain PSKU Forecast Vol', 'Norms SOH', 'Norm Days',
       'Safety Stock Vol', 'Actual Closing SOH Vol',
       'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol',
       'Assumed Closing SOH Vol', 'Assumed Closing SOH Lag 1 Vol',
       'Final Assumed Closing SOH Vol', 'Final Assumed Closing SOH Lag 1 Vol',
       'Primary Actuals Vol', 'Sec Actuals Vol', 'Primary P3M redundant Vol',
       'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M Vol',
       'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol',
       'Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lag 1 Vol',
       'LY Primary Actuals Lag 2 Vol', 'LY Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Lead 1 Vol', 'LY Primary Actuals Lead 2 Vol',
       'Off

In [1076]:
remain = remain.merge(
    brand_md_df.rename(
        columns={'brand_code': 'Brand'}
    ),
    on='Brand',
    how='left'
)
remain

,Key,Run Month,Offtake Actuals Lag 1 Vol,Offtake Actuals Lag 2 Vol,Offtake Actuals Lag 3 Vol,PSKU,Brand,Index Rate,Offtake Actuals Lag 1 Val,Offtake Actuals Lag 2 Val,Offtake Actuals Lag 3 Val,portfolio
0,Amazon RK_710542,2026-08-31,0.0000,0.0000,0.0000,710542,PCNO(R),3.492740e+05,0.000000,0.000000,0.000000,CNO
1,Amazon RK_718301,2026-08-31,0.0000,0.0000,0.0000,718301,PCNO FLEX,2.300000e+05,0.000000,0.000000,0.000000,CNO
2,Amazon RK_718458,2026-08-31,0.0000,0.0000,0.0000,718458,NHR-SABDM,1.884937e+02,0.000000,0.000000,0.000000,HAIR OILS
3,Amazon RK_718466,2026-08-31,0.0000,0.0000,0.0000,718466,PCNO(R),3.492740e+05,0.000000,0.000000,0.000000,CNO
4,Amazon RK_718477,2026-08-31,0.0000,0.0000,0.0000,718477,NHR-UTTAM,2.500000e+05,0.000000,0.000000,0.000000,CNO
...,...,...,...,...,...,...,...,...,...,...,...,...
524,Purplle_807725,2026-08-31,0.0000,0.0000,0.0000,807725,BIO OILS,3.790534e+03,0.000000,0.000000,0.000000,SKIN CARE
525,Purplle_809042,2026-08-31,0.0000,0.0000,0.0000,809042,PABABY_GM,3.664850e+02,0.000000,0.000000,0.000000,SKIN CARE
526,Purplle_809250,2026-08-31,0.0005,0.0014,0.0009,809250,LVNPST_ML,2.892133e+06,0.000145,0.000405,0.000260,PREM. HAIR NOUR.
527,Purplle_810673,2026-08-31,0.1680,0.0560,0.0980,810673,PA_ESS_HO,1.286063e+04,0.000216,0.000072,0.000126,HAIR OILS


In [ ]:
remain['Chain'] = remain['Key'].apply(lambda x: x.split('_')[0])
remain.rename(columns={'portfolio': 'Portfolio'}, inplace=True)
remain['Month Date'] = remain['Run Month'] + MonthEnd(1)

In [1079]:
cccc = final_df.copy()
final_df.shape

(60920, 91)

In [ ]:
# final_df = cccc.copy()

In [1090]:
final_df = pd.concat([final_df, remain], ignore_index=True)
final_df

,Key,Chain,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,M month,Primary Till Date Actuals Vol,...,LY Offtake P3M Val,Offtake Actuals Lag 1 Val,Offtake Actuals Lag 2 Val,Offtake Actuals Lag 3 Val,LY Offtake Actuals Lag 1 Val,LY Offtake Actuals Lag 2 Val,LY Offtake Actuals Lag 3 Val,LY Offtake Actuals Lead 1 Val,LY Offtake Actuals Lead 2 Val,Calculated Primary Val
0,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1.220081e+03,SKIN CARE,2026-08-31,2026-07-31,NaN,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
1,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1.220081e+03,SKIN CARE,2026-08-31,2026-08-31,M,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
2,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1.220081e+03,SKIN CARE,2026-08-31,2026-09-30,M+1,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
3,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1.220081e+03,SKIN CARE,2026-08-31,2026-10-31,M+2,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
4,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1.220081e+03,SKIN CARE,2026-08-31,2026-11-30,M+3,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61444,Purplle_807725,Purplle,807725,BIO OILS,3.790534e+03,SKIN CARE,2026-08-31,2026-09-30,NaN,NaN,...,NaN,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
61445,Purplle_809042,Purplle,809042,PABABY_GM,3.664850e+02,SKIN CARE,2026-08-31,2026-09-30,NaN,NaN,...,NaN,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
61446,Purplle_809250,Purplle,809250,LVNPST_ML,2.892133e+06,PREM. HAIR NOUR.,2026-08-31,2026-09-30,NaN,NaN,...,NaN,0.000145,0.000405,0.000260,NaN,NaN,NaN,NaN,NaN,NaN
61447,Purplle_810673,Purplle,810673,PA_ESS_HO,1.286063e+04,HAIR OILS,2026-08-31,2026-09-30,NaN,NaN,...,NaN,0.000216,0.000072,0.000126,NaN,NaN,NaN,NaN,NaN,NaN


In [1093]:
final_df['Chain'].unique()

array(['Amazon ARIPL', 'Amazon RK', 'Big Basket', 'Flipkart Grocery',
       'Flipkart National', 'Meesho', 'Myntra', 'Nykaa', 'Purplle'],
      dtype=object)

In [1094]:
final_depot_psku_df[final_depot_psku_df['Month Date']=='2026-09-30']['LY Primary Actuals Val'].sum()

43.479482483713

In [1095]:
final_depot_psku_df.columns

Index(['Key', 'Depot', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Calculated PSKU Primary Vol', 'Primary Actuals Vol',
       'Primary Plan Vol', 'Secondary Plan Vol', 'Secondary Actuals Vol',
       'Primary P3M Vol', 'Primary Actuals Lag 1 Vol',
       'Primary Actuals Lag 2 Vol', 'Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Vol', 'LY Primary Actuals Lag 1 Vol',
       'LY Primary Actuals Lag 2 Vol', 'LY Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Lead 1 Vol', 'LY Primary Actuals Lead 2 Vol',
       'LY Primary P3M Vol', 'Primary P3M copy Vol',
       'PSKU Primary P3M Sum Vol', 'PSKU P3M Contribution',
       'Calculated Depot PSKU Primary Vol', 'Index Rate',
       'Calculated PSKU Primary Val', 'Primary Actuals Val',
       'Primary Plan Val', 'Secondary Plan Val', 'Secondary Actuals Val',
       'Primary P3M Val', 'Primary Actuals Lag 1 Val',
       'Primary Actuals Lag 2 Val', 'Primary Actuals Lag 3 Val',
       'LY Primary Actuals Val', 'LY Primar

In [1100]:
final_df[final_df.select_dtypes(include=['number']).columns] = final_df.select_dtypes(include=['number']).fillna(0)

final_df.isnull().sum()

Key                              0
Chain                            0
PSKU                             0
Brand                            0
Index Rate                       0
                                ..
LY Offtake Actuals Lag 2 Val     0
LY Offtake Actuals Lag 3 Val     0
LY Offtake Actuals Lead 1 Val    0
LY Offtake Actuals Lead 2 Val    0
Calculated Primary Val           0
Length: 91, dtype: int64

## SAVE

In [1096]:
VERSION = 'Aug26 Live Run' 

In [1097]:
os.makedirs(f'FINAL ECOM Chain PSKU OTP/{VERSION}')

In [ ]:
final_depot_psku_df.to_csv(f'FINAL ECOM Chain PSKU OTP/{VERSION}/ECOM Depot PSKU Primary_{VERSION}.csv', index=False)
final_df.to_csv(f'FINAL ECOM Chain PSKU OTP/{VERSION}/ECOM Chain PSKU Primary_{VERSION}.csv', index=False)

In [311]:
norm_days.to_csv(f'FINAL ECOM Chain PSKU OTP/{VERSION}/Norm Days {VERSION}.csv', index=False)
norms.to_csv(f'FINAL ECOM Chain PSKU OTP/{VERSION}/Norms {VERSION}.csv', index=False)

In [312]:
soh_df.to_csv(f'FINAL ECOM Chain PSKU OTP/{VERSION}/SOH {VERSION}.csv', index=False)